<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/10_NeuroFHIR_QC_Competition_Artifacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/10_NeuroFHIR_QC_Competition_Artifacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Notebook 10
## AMIA/HL7 FHIR App Competition Artifacts and Submission Package

Run this notebook **from top to bottom in a fresh Colab runtime**.

This notebook converts the completed technical evidence from Notebooks 04–09 into a reviewer-ready competition package. It does not invent a deployed app, real users, human usability results, clinical validation, or category eligibility.

### Official 2026 submission fields packaged here

- title and category decision record;
- 1,000-character abstract;
- 3,500-character rationale, impact, and innovation response;
- 7,000-character design and implementation response;
- 3,500-character evaluation and sustainability response;
- intended users;
- 140-character project summary;
- three 500-character FHIR responses;
- 1,500-character additional-information response;
- FHIR resource list and app CapabilityStatement;
- measured-results table;
- eight-minute presentation script;
- slide and video storyboard;
- screenshot shot list;
- verified claims and limitations;
- category eligibility records;
- evidence manifest, SHA-256 inventory, and ZIP archive.

### Current official timing recorded in the package

- Submission deadline: **September 10, 2026 at 11:59 p.m. ET**
- Finalist presentation: **November 10, 2026**
- Presentation length: **approximately eight minutes**

### Eligibility boundary

The official rules state that:

- a non-student app must currently be used in real-world practice;
- a student app must have a minimum viable prototype;
- a student submission requires a signed primary-advisor attestation.

Notebook 10 therefore packages the technical evidence while keeping the final category and eligibility decision explicit. It will not mark the portal package ready unless the selected category has the required support.

In [7]:
# Cell 1 — Mount Drive, load Notebook 09 evidence, and define official submission constraints

from __future__ import annotations

import csv
import hashlib
import html
import json
import math
import os
import re
import shutil
import textwrap
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
NOTEBOOK_FILENAME = "10_NeuroFHIR_QC_Competition_Artifacts.ipynb"
NOTEBOOK_SAVE_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_FILENAME

PROJECT_CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"

NB09_AUDIT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_09_evaluation_audit.json"
)
NB09_ROOT = (
    PROJECT_ROOT
    / "evaluation/results/notebook_09_evaluation"
)
NB09_SCORECARD_PATH = (
    NB09_ROOT / "competition_evaluation_scorecard.json"
)
NB09_SEGMENTATION_PATH = (
    NB09_ROOT / "segmentation_evaluation.json"
)
NB09_ROBUSTNESS_PATH = (
    NB09_ROOT / "robustness_qc_evaluation.json"
)
NB09_LONGITUDINAL_PATH = (
    NB09_ROOT / "longitudinal_evaluation.json"
)
NB09_INTEROPERABILITY_PATH = (
    NB09_ROOT / "fhir_interoperability_evaluation.json"
)
NB09_WORKFLOW_PATH = (
    NB09_ROOT / "workflow_safety_evaluation.json"
)
NB09_TIMING_PATH = (
    NB09_ROOT / "timing_evaluation.json"
)
NB09_USABILITY_PATH = (
    NB09_ROOT / "usability_readiness.json"
)
NB09_REPRODUCIBILITY_PATH = (
    NB09_ROOT / "reproducibility_evaluation.json"
)

NB07_ROOT = (
    PROJECT_ROOT
    / "evaluation/results/notebook_07_fhir_evidence"
)
NB08_ROOT = (
    PROJECT_ROOT
    / "evaluation/results/notebook_08_human_review"
)
NB07_SUBMISSION_ROOT = (
    PROJECT_ROOT / "submission/fhir_resources/notebook_07"
)
NB08_SUBMISSION_ROOT = (
    PROJECT_ROOT / "submission/fhir_resources/notebook_08"
)

DOC_ROOT = PROJECT_ROOT / "docs"
PACKAGE_ROOT = (
    PROJECT_ROOT / "submission/amia_2026"
)
FORM_ROOT = PACKAGE_ROOT / "01_submission_form"
FHIR_ROOT = PACKAGE_ROOT / "02_fhir"
EVIDENCE_ROOT = PACKAGE_ROOT / "03_evidence"
DEMO_ROOT = PACKAGE_ROOT / "04_demo"
VISUAL_ROOT = PACKAGE_ROOT / "05_visuals"
ELIGIBILITY_ROOT = PACKAGE_ROOT / "06_eligibility"
RELEASE_ROOT = PACKAGE_ROOT / "07_release"

for folder in (
    PACKAGE_ROOT,
    FORM_ROOT,
    FHIR_ROOT,
    EVIDENCE_ROOT,
    DEMO_ROOT,
    VISUAL_ROOT,
    ELIGIBILITY_ROOT,
    RELEASE_ROOT,
):
    folder.mkdir(parents=True, exist_ok=True)

FORM_DRAFT_JSON = (
    FORM_ROOT / "AMIA_2026_SUBMISSION_FORM_DRAFT.json"
)
FORM_DRAFT_MD = (
    FORM_ROOT / "AMIA_2026_SUBMISSION_FORM_DRAFT.md"
)
CHARACTER_AUDIT_CSV = (
    FORM_ROOT / "submission_character_limit_audit.csv"
)
RESOURCE_LIST_PATH = (
    FHIR_ROOT / "NeuroFHIR_QC_FHIR_RESOURCE_LIST.md"
)
APP_CAPABILITY_PATH = (
    FHIR_ROOT / "neurofhir_qc_app_capability_statement.json"
)
PROVENANCE_CHECKLIST_CSV = (
    EVIDENCE_ROOT / "provenance_completeness_checklist.csv"
)
MEASURED_RESULTS_CSV = (
    EVIDENCE_ROOT / "measured_results_table.csv"
)
MEASURED_RESULTS_MD = (
    EVIDENCE_ROOT / "MEASURED_RESULTS_TABLE.md"
)
DEMO_SCRIPT_PATH = (
    DEMO_ROOT / "EIGHT_MINUTE_PRESENTATION_SCRIPT.md"
)
RUN_OF_SHOW_PATH = (
    DEMO_ROOT / "LIVE_DEMO_RUN_OF_SHOW.md"
)
VIDEO_STORYBOARD_PATH = (
    DEMO_ROOT / "DEMO_VIDEO_STORYBOARD.md"
)
SLIDE_OUTLINE_PATH = (
    DEMO_ROOT / "SLIDE_DECK_OUTLINE.md"
)
SCREENSHOT_LIST_PATH = (
    DEMO_ROOT / "SCREENSHOT_SHOT_LIST.md"
)
PORTAL_CLAIMS_PATH = (
    FORM_ROOT / "PORTAL_READY_VERIFIED_CLAIMS.md"
)
CATEGORY_DECISION_PATH = (
    ELIGIBILITY_ROOT / "CATEGORY_ELIGIBILITY_DECISION.md"
)
STUDENT_ATTESTATION_TEMPLATE_PATH = (
    ELIGIBILITY_ROOT / "STUDENT_ADVISOR_ATTESTATION_TEMPLATE.md"
)
NONSTUDENT_EVIDENCE_TEMPLATE_PATH = (
    ELIGIBILITY_ROOT / "NONSTUDENT_REAL_WORLD_USE_EVIDENCE_TEMPLATE.md"
)
FINAL_CHECKLIST_PATH = (
    RELEASE_ROOT / "FINAL_SUBMISSION_CHECKLIST.md"
)
PACKAGE_README_PATH = (
    PACKAGE_ROOT / "README_SUBMISSION_PACKAGE.md"
)
HTML_SUMMARY_PATH = (
    PACKAGE_ROOT / "index.html"
)
ARTIFACT_MANIFEST_PATH = (
    RELEASE_ROOT / "artifact_manifest.json"
)
CHECKSUM_PATH = (
    RELEASE_ROOT / "SHA256SUMS.txt"
)
ZIP_PATH = (
    PROJECT_ROOT
    / "submission/NeuroFHIR_QC_AMIA_2026_Submission_Package.zip"
)
AUDIT_JSON_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_10_competition_artifacts_audit.json"
)
AUDIT_MD_PATH = (
    DOC_ROOT / "NOTEBOOK_10_COMPETITION_ARTIFACTS.md"
)

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        handle.write("\n")
    temporary.replace(path)

def utc_now() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def clean_text(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip()

def safe_float(value: Any, default: float = float("nan")) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

required_paths = [
    PROJECT_CONFIG_PATH,
    NOTEBOOK_MANIFEST_PATH,
    NB09_AUDIT_PATH,
    NB09_SCORECARD_PATH,
    NB09_SEGMENTATION_PATH,
    NB09_ROBUSTNESS_PATH,
    NB09_LONGITUDINAL_PATH,
    NB09_INTEROPERABILITY_PATH,
    NB09_WORKFLOW_PATH,
    NB09_TIMING_PATH,
    NB09_USABILITY_PATH,
    NB09_REPRODUCIBILITY_PATH,
]
missing = [
    str(path)
    for path in required_paths
    if not path.exists() or path.stat().st_size == 0
]
if missing:
    raise FileNotFoundError(
        "Notebook 10 prerequisites are missing:\n"
        + "\n".join(f" - {path}" for path in missing)
    )

project_config = load_json(PROJECT_CONFIG_PATH)
notebook_manifest = load_json(NOTEBOOK_MANIFEST_PATH)
nb09_audit = load_json(NB09_AUDIT_PATH)
scorecard = load_json(NB09_SCORECARD_PATH)
segmentation = load_json(NB09_SEGMENTATION_PATH)
robustness = load_json(NB09_ROBUSTNESS_PATH)
longitudinal = load_json(NB09_LONGITUDINAL_PATH)
interoperability = load_json(NB09_INTEROPERABILITY_PATH)
workflow = load_json(NB09_WORKFLOW_PATH)
timing = load_json(NB09_TIMING_PATH)
usability = load_json(NB09_USABILITY_PATH)
reproducibility = load_json(NB09_REPRODUCIBILITY_PATH)

if nb09_audit.get("status") != "completed":
    raise RuntimeError("Notebook 09 audit is not completed.")

metrics_09 = nb09_audit.get("metrics", {})
required_nb09_metrics = {
    "segmentation_case_count": 3,
    "standard_perturbation_run_count": 12,
    "severe_challenge_run_count": 1,
    "low_confidence_detection_rate": 1.0,
    "scenario_alignment_rate": 1.0,
    "fhir_validation_pass_rate": 1.0,
    "fhir_transaction_success_rate": 1.0,
    "fhir_transaction_entry_success_rate": 1.0,
    "fhir_critical_field_preservation_rate": 1.0,
    "human_review_transition_success_rate": 1.0,
    "low_confidence_finalization_block_rate": 1.0,
}
for key, expected in required_nb09_metrics.items():
    actual = metrics_09.get(key)
    if actual != expected:
        raise AssertionError(
            f"Notebook 09 gate failed: {key}={actual!r}, "
            f"expected {expected!r}."
        )

OFFICIAL_RULES_URL = (
    "https://amia.org/education-events/"
    "amia-2026-annual-symposium/fhir"
)
OFFICIAL_DEADLINE = "2026-09-10 23:59 ET"
OFFICIAL_PRESENTATION_DATE = "2026-11-10"
OFFICIAL_PRESENTATION_WINDOW = "09:45–11:00 CST"
OFFICIAL_PRESENTATION_LENGTH = "approximately 8 minutes"

CHARACTER_LIMITS = {
    "project_abstract": 1000,
    "rationale_impact_innovation": 3500,
    "design_implementation": 7000,
    "evaluation_sustainability": 3500,
    "twitter_summary": 140,
    "fhir_use": 500,
    "fhir_release_resources": 500,
    "fhir_data_source_access": 500,
    "other_information": 1500,
}

REPOSITORY_URL = os.getenv(
    "NEUROFHIR_QC_REPOSITORY_URL",
    "https://github.com/SANGHATI23/neurofhir-qc",
).strip()
APP_OR_DEMO_URL = os.getenv(
    "NEUROFHIR_QC_APP_OR_DEMO_URL",
    "",
).strip()
SUBMISSION_CATEGORY = os.getenv(
    "NEUROFHIR_QC_SUBMISSION_CATEGORY",
    "UNRESOLVED",
).strip().title()
SUBMITTER_NAME = os.getenv(
    "NEUROFHIR_QC_SUBMITTER_NAME",
    "Sanghati Basu",
).strip()
AFFILIATION = os.getenv(
    "NEUROFHIR_QC_AFFILIATION",
    "VERIFY CURRENT AFFILIATION",
).strip()
CONCEPTION_DATE = os.getenv(
    "NEUROFHIR_QC_CONCEPTION_DATE",
    "",
).strip()
IMPLEMENTATION_DATE = os.getenv(
    "NEUROFHIR_QC_IMPLEMENTATION_DATE",
    str(project_config.get("created_utc", ""))[:10],
).strip()
SMART_GALLERY_AGREEMENT = os.getenv(
    "NEUROFHIR_QC_SMART_GALLERY_AGREEMENT",
    "UNRESOLVED",
).strip()
STUDENT_ADVISOR_ATTESTATION_AVAILABLE = (
    os.getenv(
        "NEUROFHIR_QC_STUDENT_ATTESTATION_AVAILABLE",
        "false",
    ).strip().lower()
    in {"1", "true", "yes", "y"}
)
REAL_WORLD_PRACTICE_EVIDENCE_AVAILABLE = (
    os.getenv(
        "NEUROFHIR_QC_REAL_WORLD_USE_EVIDENCE_AVAILABLE",
        "false",
    ).strip().lower()
    in {"1", "true", "yes", "y"}
)

LOGO_PATH = Path(
    os.getenv(
        "NEUROFHIR_QC_LOGO_PATH",
        str(PROJECT_ROOT / "submission/branding/logo.png"),
    )
)
HEADSHOT_PATH = Path(
    os.getenv(
        "NEUROFHIR_QC_HEADSHOT_PATH",
        str(PROJECT_ROOT / "submission/branding/headshot.jpg"),
    )
)
PROMOTIONAL_PHOTO_PATH = Path(
    os.getenv(
        "NEUROFHIR_QC_PROMOTIONAL_PHOTO_PATH",
        str(PROJECT_ROOT / "submission/branding/promotional_photo.png"),
    )
)

print("=" * 112)
print("✅ Notebook 09 completion gate passed")
print("✅ Official 2026 submission fields and character limits loaded")
print(f"📅 Submission deadline: {OFFICIAL_DEADLINE}")
print(f"🎤 Presentation: {OFFICIAL_PRESENTATION_DATE}, {OFFICIAL_PRESENTATION_WINDOW}")
print(f"⏱️ Presentation length: {OFFICIAL_PRESENTATION_LENGTH}")
print(f"📂 Competition package root: {PACKAGE_ROOT}")
print(f"⚠️ Submission category: {SUBMISSION_CATEGORY}")
print("⚠️ Category eligibility will be evaluated, not assumed")
print("=" * 112)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Notebook 09 completion gate passed
✅ Official 2026 submission fields and character limits loaded
📅 Submission deadline: 2026-09-10 23:59 ET
🎤 Presentation: 2026-11-10, 09:45–11:00 CST
⏱️ Presentation length: approximately 8 minutes
📂 Competition package root: /content/drive/MyDrive/neurofhir-qc/submission/amia_2026
⚠️ Submission category: Unresolved
⚠️ Category eligibility will be evaluated, not assumed


In [8]:
# Cell 2 — Consolidate the executed evidence into one competition source of truth

mean_dice = safe_float(
    segmentation.get("mean_whole_tumor_dice")
)
mean_abs_volume_error_ml = safe_float(
    segmentation.get("mean_absolute_volume_error_ml")
)
mean_inference_seconds = safe_float(
    segmentation.get("mean_inference_seconds")
)
case_count = int(segmentation.get("case_count", 0))

standard_perturbation_types = int(
    robustness.get(
        "standard_perturbation_type_count",
        0,
    )
)
standard_perturbation_runs = int(
    robustness.get(
        "standard_perturbation_run_count",
        0,
    )
)
challenge_runs = int(
    robustness.get("challenge_run_count", 0)
)
low_confidence_detection_rate = safe_float(
    robustness.get(
        "low_confidence_manual_review_detection_rate"
    )
)

alignment_pass_count = int(
    longitudinal.get(
        "scenario_alignment_pass_count",
        0,
    )
)
alignment_rate = safe_float(
    longitudinal.get("scenario_alignment_rate")
)
withholding_rate = safe_float(
    longitudinal.get(
        "low_confidence_interpretation_withholding_rate"
    )
)

validation_targets = int(
    interoperability.get(
        "validation_target_count",
        0,
    )
)
validation_pass_count = int(
    interoperability.get(
        "validation_pass_count",
        0,
    )
)
transaction_count = int(
    interoperability.get("transaction_count", 0)
)
transaction_success_count = int(
    interoperability.get(
        "transaction_success_count",
        0,
    )
)
submitted_entry_count = int(
    interoperability.get(
        "submitted_entry_count",
        0,
    )
)
successful_entry_count = int(
    interoperability.get(
        "successful_entry_count",
        0,
    )
)
readback_resource_count = int(
    interoperability.get(
        "generated_evidence_readback_resource_count",
        0,
    )
)
reference_graph_edge_count = int(
    interoperability.get(
        "reference_graph_edge_count",
        0,
    )
)
critical_field_preservation_rate = safe_float(
    interoperability.get(
        "critical_field_preservation_rate"
    )
)

review_transition_event_count = int(
    workflow.get(
        "review_transition_event_count",
        0,
    )
)
accepted_case_count = int(
    workflow.get("accepted_case_count", 0)
)
correction_required_case_count = int(
    workflow.get(
        "correction_required_case_count",
        0,
    )
)
rejected_case_count = int(
    workflow.get("rejected_case_count", 0)
)
review_provenance_event_count = int(
    workflow.get(
        "review_provenance_event_count",
        0,
    )
)
finalization_block_rate = safe_float(
    workflow.get(
        "low_confidence_finalization_block_rate"
    )
)

expected_values = {
    "case_count": (case_count, 3),
    "standard_perturbation_types": (
        standard_perturbation_types,
        4,
    ),
    "standard_perturbation_runs": (
        standard_perturbation_runs,
        12,
    ),
    "challenge_runs": (challenge_runs, 1),
    "alignment_pass_count": (
        alignment_pass_count,
        3,
    ),
    "review_transition_event_count": (
        review_transition_event_count,
        4,
    ),
    "accepted_case_count": (
        accepted_case_count,
        2,
    ),
    "correction_required_case_count": (
        correction_required_case_count,
        1,
    ),
    "rejected_case_count": (
        rejected_case_count,
        1,
    ),
    "review_provenance_event_count": (
        review_provenance_event_count,
        4,
    ),
}
for key, (actual, expected) in expected_values.items():
    if actual != expected:
        raise AssertionError(
            f"Competition evidence mismatch: {key}={actual}, "
            f"expected {expected}."
        )

required_rates = {
    "low_confidence_detection_rate":
        low_confidence_detection_rate,
    "alignment_rate": alignment_rate,
    "withholding_rate": withholding_rate,
    "critical_field_preservation_rate":
        critical_field_preservation_rate,
    "finalization_block_rate": finalization_block_rate,
}
for key, value in required_rates.items():
    if not math.isclose(
        value,
        1.0,
        rel_tol=0,
        abs_tol=1e-9,
    ):
        raise AssertionError(
            f"Competition safety metric failed: {key}={value}"
        )

if validation_pass_count != validation_targets:
    raise AssertionError(
        "FHIR server validation is not 100%."
    )
if transaction_success_count != transaction_count:
    raise AssertionError(
        "FHIR transaction success is not 100%."
    )
if successful_entry_count != submitted_entry_count:
    raise AssertionError(
        "FHIR transaction-entry success is not 100%."
    )

competition_evidence = {
    "project_name": "NeuroFHIR-QC",
    "display_title": (
        "NeuroFHIR-QC: Trustworthy Longitudinal "
        "Neuroimaging AI in FHIR"
    ),
    "one_sentence_pitch": (
        "NeuroFHIR-QC transforms AI-generated longitudinal "
        "brain-tumor or lesion measurements into quality-scored, "
        "provenance-aware, human-reviewed FHIR clinical-research results."
    ),
    "positioning": (
        "Academic research implementation evaluated with public "
        "de-identified neuroimaging data and synthetic FHIR R4 records."
    ),
    "data_boundary": {
        "public_deidentified_research_mri_only": True,
        "synthetic_fhir_only": True,
        "real_patient_data_used": False,
        "clinical_deployment_claimed": False,
    },
    "segmentation": {
        "case_count": case_count,
        "mean_whole_tumor_dice": mean_dice,
        "mean_absolute_volume_error_ml":
            mean_abs_volume_error_ml,
        "mean_inference_seconds": mean_inference_seconds,
    },
    "robustness": {
        "controlled_perturbation_type_count":
            standard_perturbation_types,
        "standard_run_count":
            standard_perturbation_runs,
        "severe_challenge_run_count": challenge_runs,
        "low_confidence_detection_rate":
            low_confidence_detection_rate,
    },
    "longitudinal": {
        "calculation_case_count": 3,
        "scenario_alignment_pass_count":
            alignment_pass_count,
        "scenario_alignment_rate": alignment_rate,
        "low_confidence_interpretation_withholding_rate":
            withholding_rate,
    },
    "fhir": {
        "release": "FHIR R4 / 4.0.1",
        "validation_pass_count": validation_pass_count,
        "validation_target_count": validation_targets,
        "transaction_success_count":
            transaction_success_count,
        "transaction_count": transaction_count,
        "successful_entry_count":
            successful_entry_count,
        "submitted_entry_count":
            submitted_entry_count,
        "readback_resource_count":
            readback_resource_count,
        "critical_field_preservation_rate":
            critical_field_preservation_rate,
        "reference_graph_edge_count":
            reference_graph_edge_count,
    },
    "human_review": {
        "review_transition_event_count":
            review_transition_event_count,
        "accepted_case_count": accepted_case_count,
        "correction_required_case_count":
            correction_required_case_count,
        "rejected_case_count": rejected_case_count,
        "review_provenance_event_count":
            review_provenance_event_count,
        "low_confidence_finalization_block_rate":
            finalization_block_rate,
        "real_clinician_review_performed": False,
    },
    "usability": {
        "technical_task_path_count": int(
            usability.get("technical_task_count", 0)
        ),
        "human_usability_study_performed": bool(
            usability.get(
                "human_usability_study_performed",
                False,
            )
        ),
    },
    "open_evaluation_gaps": scorecard.get(
        "open_gaps",
        [],
    ),
}
write_json(
    EVIDENCE_ROOT / "competition_evidence_source_of_truth.json",
    competition_evidence,
)

print("=" * 112)
print("✅ Competition source of truth created")
print(
    f"✅ Segmentation: {case_count} cases, "
    f"mean Dice {mean_dice:.4f}"
)
print(
    f"✅ Robustness: {standard_perturbation_runs} standard + "
    f"{challenge_runs} challenge runs"
)
print("✅ Longitudinal scenario alignment: 3/3")
print(
    f"✅ FHIR validation: "
    f"{validation_pass_count}/{validation_targets}"
)
print(
    f"✅ FHIR transactions: "
    f"{transaction_success_count}/{transaction_count}"
)
print(
    f"✅ FHIR transaction entries: "
    f"{successful_entry_count}/{submitted_entry_count}"
)
print(
    f"✅ Human-review transitions: "
    f"{review_transition_event_count}"
)
print("✅ Unsupported clinical and usability claims remain blocked")
print("=" * 112)

✅ Competition source of truth created
✅ Segmentation: 3 cases, mean Dice 0.9014
✅ Robustness: 12 standard + 1 challenge runs
✅ Longitudinal scenario alignment: 3/3
✅ FHIR validation: 34/34
✅ FHIR transactions: 7/7
✅ FHIR transaction entries: 79/79
✅ Human-review transitions: 4
✅ Unsupported clinical and usability claims remain blocked


In [9]:
# Cell 3 — Generate the official AMIA 2026 submission-form draft and character audit

title = competition_evidence["display_title"]

project_abstract = clean_text(
    f"""
    NeuroFHIR-QC addresses the missing interoperability layer between
    computational neuroimaging AI and reviewable clinical-research data.
    It links public de-identified MRI outputs to synthetic FHIR R4 patient
    context, calculates tumor or lesion volume and longitudinal change,
    scores perturbation stability and provenance completeness, and keeps
    every AI result preliminary until explicit review. Stable and
    progression cases can be accepted; an unstable case is routed through
    correction-required and rejection states with Task and Provenance
    records. In three executable demonstration cases, mean whole-tumor
    Dice was {mean_dice:.4f}; all {validation_targets} FHIR validation
    targets, {transaction_count} transactions, and
    {submitted_entry_count} transaction entries passed, with 100% critical
    field preservation on read-back. The prototype uses only public
    de-identified imaging and synthetic FHIR records and does not claim
    clinical validation or deployment.
    """
)

rationale_impact_innovation = clean_text(
    """
    Computational neuroimaging results are frequently produced as NIfTI
    files, segmentation masks, notebook outputs, JSON or CSV measurements,
    and free-text review notes. These artifacts may be useful for research,
    but they are often disconnected from patient context, source-study
    identity, model and preprocessing versions, quality-control evidence,
    longitudinal history, and structured reviewer decisions. The affected
    audience includes neuroimaging researchers, biomedical informaticians,
    AI developers, data stewards, and clinical-research teams that need to
    reuse or audit model-derived measurements.

    NeuroFHIR-QC treats the interoperability and governance gap—not merely
    segmentation accuracy—as the central problem. It turns a model output
    into a governed evidence object linked to Patient, Condition,
    ImagingStudy, Observation, DiagnosticReport, Device, Provenance, Task,
    Practitioner, and transaction Bundle resources. The result can be
    inspected, validated, written to a FHIR server, read back, and traced
    from source image through model execution and human-review state.

    The main innovation is the combination of longitudinal volumetry,
    perturbation-based quality triage, explicit preliminary status, and
    FHIR-native human oversight. The strongest demonstration is a severe
    instability case: the numerical result remains available for audit,
    its longitudinal interpretation is withheld, autonomous finalization
    is blocked, a correction-required state is preserved, and the original
    AI result is rejected and marked entered-in-error with complete review
    Provenance.

    The near-term impact is a reproducible research prototype showing how
    neuro-AI outputs can become standardized and reviewable rather than
    isolated files. The longer-term opportunity is a reusable translation
    layer for multi-site neuroimaging studies, model monitoring, and
    clinically linked research workflows. The present evidence is based on
    three public research MRI cases and synthetic FHIR R4 context; it does
    not establish diagnostic performance, patient benefit, or production
    deployment.
    """
)

design_implementation = clean_text(
    f"""
    NeuroFHIR-QC was implemented as a closed, auditable pipeline with
    persistent artifacts under a versioned repository. The data boundary
    is public de-identified brain MRI plus synthetic FHIR R4 records. Three
    locked demonstration behaviors were created: stable longitudinal
    volume, progression, and severe low-confidence failure.

    Synthetic FHIR context provides Patient, Condition, and ImagingStudy
    resources plus a prior reviewed Observation. A MONAI-based research
    segmentation workflow produces a current whole-tumor or lesion volume,
    mask evidence, model identity, software version, and runtime metadata.
    Volumetry is computed from the predicted mask and image geometry.
    Segmentation evaluation includes Dice, sensitivity, precision, HD95,
    absolute volume error, relative volume error, and inference time.

    A trust layer executes four controlled perturbation types across all
    three cases, plus one severe low-confidence challenge. It combines
    perturbation agreement, volume stability, plausibility, provenance
    completeness, and threshold certainty into an engineering workflow
    score. Categories are High confidence, Review recommended, and Manual
    review required. The score is explicitly not a calibrated probability
    of clinical safety.

    Longitudinal logic compares the prior reviewed volume with the current
    AI-derived volume. The stable and progression cases produce display
    interpretations, while the low-confidence case retains the numerical
    calculation for audit but withholds the interpretation. All model
    Observations begin as preliminary, and autonomous finalization is
    disabled.

    The FHIR evidence builder creates linked Device, Observation,
    DiagnosticReport, Task, and Provenance resources. Transaction Bundles
    include complete synthetic source context and use deterministic PUT
    requests. Same-Bundle references use deterministic URN fullUrls. The
    prototype queries a public HAPI FHIR R4 sandbox, reads the
    CapabilityStatement, submits resources to $validate, writes transaction
    Bundles, reads resources back, checks critical-field preservation, and
    archives OperationOutcome, transaction-response, network timing, and
    reference-graph evidence.

    Human review is represented as an explicit state machine. Two
    high-confidence cases move from preliminary/requested to
    final/completed. The unstable case moves first to
    correction-required/on-hold, that intermediate state is read back, and
    the uncorrected original result is then rejected and marked
    entered-in-error. Four human-review Provenance events capture reviewer
    role, timestamp, reason, note, decision, and affected resources. The
    reviewer is an explicitly synthetic research reviewer; no clinician
    usability or clinical judgment is claimed.

    The major implementation challenges were preserving a truthful stable
    longitudinal demonstration, correcting FHIR R4 element choices,
    resolving transaction-Bundle references, and preventing stale
    downstream artifacts. These were addressed with deterministic
    synthetic-context realignment, R4-specific structures, URN reference
    rewriting, same-runtime execution gates, file checksums, completion
    audits, and read-back verification.

    Measured execution produced a mean whole-tumor Dice of
    {mean_dice:.4f} across three cases, {standard_perturbation_runs}
    standard perturbation runs and {challenge_runs} severe challenge,
    three of three longitudinal scenario alignments,
    {validation_pass_count}/{validation_targets} server-validation passes,
    {transaction_success_count}/{transaction_count} successful
    transactions, and {successful_entry_count}/{submitted_entry_count}
    successful transaction entries. The implementation remains a research
    minimum viable prototype rather than a hospital-deployed application.
    """
)

evaluation_sustainability = clean_text(
    f"""
    Evaluation was performed across segmentation, robustness, longitudinal
    behavior, FHIR interoperability, workflow safety, timing, and
    reproducibility. Three public research MRI cases produced a mean
    whole-tumor Dice of {mean_dice:.4f}, mean absolute volume error of
    {mean_abs_volume_error_ml:.3f} mL, and mean inference time of
    {mean_inference_seconds:.2f} seconds. These are executable
    demonstration results, not independent external validation.

    Four controlled perturbation types were run across all three cases
    ({standard_perturbation_runs} standard runs), followed by one severe
    synthetic challenge. Stable and progression cases received
    high-confidence workflow scores, while the severe challenge was
    classified Manual review required. Low-confidence detection,
    preliminary-status retention, interpretation withholding, and
    autonomous-finalization blocking were each 100%.

    Longitudinal calculations completed for three of three cases and the
    planned stable, progression, and low-confidence behaviors aligned
    three of three. Across AI-evidence and human-review stages,
    {validation_pass_count}/{validation_targets} FHIR validation targets,
    {transaction_success_count}/{transaction_count} transaction Bundles,
    and {successful_entry_count}/{submitted_entry_count} transaction
    entries passed. {readback_resource_count} generated or reviewed
    resources were read back with 100% critical-field preservation. The
    archived FHIR reference graph contains {reference_graph_edge_count}
    edges.

    The scripted review workflow accepted two high-confidence cases,
    preserved one correction-required intermediate state, rejected the
    unstable result, and created four review Provenance events. This
    demonstrates workflow mechanics, not clinician agreement. Five
    predecessor audits completed, required evaluation artifacts were
    non-empty, and checksums support reproducibility.

    Sustainability is based on an open, modular repository with reusable
    FHIR builders, service code, deterministic synthetic cases, versioned
    artifacts, validation reports, and testable notebooks. Future
    maintenance can extend biomarker adapters and replace the public
    sandbox with an institutional FHIR endpoint without changing the core
    evidence model. Honest open gaps are a finished end-user interface,
    measured interactive task time and click count, and a real human
    usability study with task completion, errors, feedback, and SUS. No
    real-user or clinical-impact result is claimed until those evaluations
    are performed.
    """
)

intended_users = clean_text(
    """
    Intended users are neuroimaging and computational neuroscience
    researchers, biomedical and clinical informaticians, medical-imaging
    AI developers, research data stewards, FHIR implementers, and review
    teams that need model-derived neuroimaging biomarkers to be
    traceable, quality-aware, longitudinally interpretable, and reusable
    in standards-based research workflows.
    """
)

twitter_summary = (
    "NeuroFHIR-QC turns neuro-AI volumes into QC-scored, "
    "human-reviewed, provenance-aware FHIR R4 evidence."
)

fhir_use = clean_text(
    """
    FHIR is the application's evidence and workflow layer, not an export
    afterthought. NeuroFHIR-QC reads synthetic patient and imaging context,
    creates preliminary AI Observations and linked DiagnosticReport,
    Device, Task, and Provenance resources, validates transaction Bundles,
    writes them to an R4 server, reads them back, and records accept,
    correction-required, and reject transitions with review Provenance.
    """
)

fhir_release_resources = clean_text(
    """
    The prototype uses FHIR R4 (4.0.1). Demonstrated resources are Patient,
    Condition, ImagingStudy, Observation, DiagnosticReport, Device,
    Provenance, Task, Practitioner, Bundle, and CapabilityStatement.
    Observation stores volume and QC components; Task represents review;
    Device identifies the model; Provenance records algorithmic generation
    and human decisions. Base R4 resources are used; US Core profiles are
    not currently claimed.
    """
)

fhir_data_source_access = clean_text(
    """
    FHIR resources are synthetic and are linked to public de-identified
    research MRI outputs. The app accesses a public HAPI FHIR R4 sandbox
    through the REST API, reads its CapabilityStatement, validates
    resources with $validate, submits deterministic transaction Bundles,
    performs direct read-back, and archives validation, response, timing,
    checksum, and reference-integrity evidence. No production EHR or real
    patient data are used.
    """
)

other_information = clean_text(
    f"""
    The repository contains an executable evidence chain from MRI
    segmentation and volumetry through robustness testing, QC-aware
    longitudinal comparison, FHIR R4 validation and transaction write-back,
    human-review transitions, read-back, and Provenance. The central safety
    rule is that every AI-generated Observation begins as preliminary and
    cannot become final without explicit review. The most important case is
    the unstable result: its numerical value is retained for audit, its
    interpretation is withheld, correction-required is preserved, and the
    uncorrected result is rejected and marked entered-in-error.

    Current measured evidence includes mean whole-tumor Dice
    {mean_dice:.4f}, {validation_pass_count}/{validation_targets} server
    validation passes, {transaction_success_count}/{transaction_count}
    successful transactions, and {successful_entry_count}/
    {submitted_entry_count} successful transaction entries. The app is a
    research MVP evaluated on three public MRI cases with synthetic FHIR
    context. It is not clinically validated, not deployed in a hospital,
    and has no paying customers or patient impact. Human usability remains
    an explicit open gap.
    """
)

submission_fields = {
    "title": title,
    "category": SUBMISSION_CATEGORY,
    "letter_of_support": (
        "Required if Student; availability="
        + str(STUDENT_ADVISOR_ATTESTATION_AVAILABLE)
    ),
    "project_abstract": project_abstract,
    "rationale_impact_innovation":
        rationale_impact_innovation,
    "design_implementation": design_implementation,
    "evaluation_sustainability":
        evaluation_sustainability,
    "intended_user_audience": intended_users,
    "twitter_summary": twitter_summary,
    "fhir_use": fhir_use,
    "fhir_release_resources":
        fhir_release_resources,
    "fhir_data_source_access":
        fhir_data_source_access,
    "fhir_resources_upload": (
        "02_fhir/neurofhir_qc_app_capability_statement.json "
        "and 02_fhir/NeuroFHIR_QC_FHIR_RESOURCE_LIST.md"
    ),
    "uses_us_core_or_other_implementation_guides": (
        "No implementation-guide conformance is currently claimed. "
        "The executed prototype uses base FHIR R4 resources."
    ),
    "fhir_technologies": (
        "FHIR R4 REST, CapabilityStatement discovery, "
        "$validate, transaction Bundles, deterministic PUT, "
        "OperationOutcome, and HAPI FHIR sandbox. "
        "SMART, CDS Hooks, Bulk FHIR, and CQL are not yet implemented."
    ),
    "other_information": other_information,
    "logo_headshot_promotional_photo": {
        "logo_available": LOGO_PATH.exists(),
        "headshot_available": HEADSHOT_PATH.exists(),
        "promotional_photo_available":
            PROMOTIONAL_PHOTO_PATH.exists(),
    },
    "paying_customers": "No",
    "conceived": (
        CONCEPTION_DATE or "TO BE VERIFIED"
    ),
    "implemented": (
        IMPLEMENTATION_DATE or "TO BE VERIFIED"
    ),
    "users_or_patients_impacted": (
        "0 clinical users and 0 patients impacted. "
        "Technical prototype evaluated on 3 synthetic cases "
        "linked to public de-identified research MRI."
    ),
    "website_url": (
        APP_OR_DEMO_URL or REPOSITORY_URL
    ),
    "repository_url": REPOSITORY_URL,
    "smart_app_gallery_agreement":
        SMART_GALLERY_AGREEMENT,
    "submitter": SUBMITTER_NAME,
    "affiliation": AFFILIATION,
}

character_rows = []
for field, limit in CHARACTER_LIMITS.items():
    value = submission_fields[field]
    count = len(value)
    character_rows.append(
        {
            "field": field,
            "character_count": count,
            "character_limit": limit,
            "remaining_characters": limit - count,
            "within_limit": count <= limit,
        }
    )

failed_character_fields = [
    row
    for row in character_rows
    if not row["within_limit"]
]
if failed_character_fields:
    raise AssertionError(
        "Submission fields exceed official character limits:\n"
        + json.dumps(
            failed_character_fields,
            indent=2,
        )
    )

write_json(
    FORM_DRAFT_JSON,
    {
        "official_rules_url": OFFICIAL_RULES_URL,
        "generated_utc": utc_now(),
        "deadline": OFFICIAL_DEADLINE,
        "submission_fields": submission_fields,
        "character_limit_audit": character_rows,
    },
)

with CHARACTER_AUDIT_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(character_rows[0].keys()),
    )
    writer.writeheader()
    writer.writerows(character_rows)

def form_section(
    heading: str,
    value: Any,
    field: str | None = None,
) -> str:
    count_text = ""
    if field in CHARACTER_LIMITS:
        count_text = (
            f"\n\n**Characters:** {len(str(value))}/"
            f"{CHARACTER_LIMITS[field]}"
        )
    if isinstance(value, (dict, list)):
        rendered = "```json\n" + json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
        ) + "\n```"
    else:
        rendered = str(value)
    return f"## {heading}\n\n{rendered}{count_text}"

form_sections = [
    "# AMIA 2026 FHIR App Competition Submission Draft",
    "",
    f"**Generated:** {utc_now()}  ",
    f"**Official deadline:** {OFFICIAL_DEADLINE}  ",
    f"**Official rules:** {OFFICIAL_RULES_URL}  ",
    "",
    "> This is a source-grounded draft. Verify category, affiliation, "
    "dates, URLs, branding, and eligibility before portal submission.",
    "",
    form_section("1. Title", title),
    form_section("2. Category", SUBMISSION_CATEGORY),
    form_section(
        "3. Student Letter of Support",
        submission_fields["letter_of_support"],
    ),
    form_section(
        "4. Project Abstract",
        project_abstract,
        "project_abstract",
    ),
    form_section(
        "5. Project Rationale, Impact and Innovation",
        rationale_impact_innovation,
        "rationale_impact_innovation",
    ),
    form_section(
        "6. Project Design and Implementation",
        design_implementation,
        "design_implementation",
    ),
    form_section(
        "7. Project Evaluation and Sustainability",
        evaluation_sustainability,
        "evaluation_sustainability",
    ),
    form_section(
        "8. Intended User/Audience",
        intended_users,
    ),
    form_section(
        "9. 140-character Project Summary",
        twitter_summary,
        "twitter_summary",
    ),
    form_section(
        "10. How FHIR Is Used",
        fhir_use,
        "fhir_use",
    ),
    form_section(
        "11. FHIR Release and Resources",
        fhir_release_resources,
        "fhir_release_resources",
    ),
    form_section(
        "12. FHIR Data Source and Access",
        fhir_data_source_access,
        "fhir_data_source_access",
    ),
    form_section(
        "13. FHIR Resources Upload",
        submission_fields["fhir_resources_upload"],
    ),
    form_section(
        "14. US Core or Other Implementation Guides",
        submission_fields[
            "uses_us_core_or_other_implementation_guides"
        ],
    ),
    form_section(
        "15. FHIR Technologies",
        submission_fields["fhir_technologies"],
    ),
    form_section(
        "16. Other Information",
        other_information,
        "other_information",
    ),
    form_section(
        "17–24. Administrative Fields",
        {
            key: submission_fields[key]
            for key in (
                "logo_headshot_promotional_photo",
                "paying_customers",
                "conceived",
                "implemented",
                "users_or_patients_impacted",
                "website_url",
                "repository_url",
                "smart_app_gallery_agreement",
                "submitter",
                "affiliation",
            )
        },
    ),
]
FORM_DRAFT_MD.write_text(
    "\n".join(form_sections).strip() + "\n",
    encoding="utf-8",
)

PORTAL_CLAIMS_PATH.write_text(
    textwrap.dedent(
        f"""
        # Portal-Ready Verified Claims

        These statements are supported by the executed repository evidence.

        - Three public de-identified research MRI demonstration cases were evaluated.
        - Mean whole-tumor Dice was {mean_dice:.4f}.
        - Mean absolute volume error was {mean_abs_volume_error_ml:.3f} mL.
        - Mean inference time was {mean_inference_seconds:.2f} seconds.
        - Four controlled perturbation types produced {standard_perturbation_runs} standard runs.
        - One severe synthetic challenge was routed to manual review.
        - Longitudinal scenario alignment was 3/3.
        - Low-confidence interpretation withholding was 100%.
        - FHIR server validation passed {validation_pass_count}/{validation_targets} targets.
        - FHIR transaction write-back passed {transaction_success_count}/{transaction_count} Bundles.
        - FHIR transaction entries passed {successful_entry_count}/{submitted_entry_count}.
        - {readback_resource_count} generated or reviewed resources were read back with 100% critical-field preservation.
        - The review workflow accepted two cases, preserved one correction-required intermediate state, and rejected one unstable result.
        - Four human-review Provenance events were created.
        - The prototype uses public de-identified imaging and synthetic FHIR R4 records only.
        - No clinical deployment, clinical validation, real clinician review, paying customers, or patient impact is claimed.
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

print("=" * 112)
print("✅ Official submission-form draft generated")
print("✅ All character-limited fields are within official limits")
for row in character_rows:
    print(
        f" - {row['field']}: "
        f"{row['character_count']}/"
        f"{row['character_limit']}"
    )
print(f"✅ Form draft: {FORM_DRAFT_MD}")
print("=" * 112)

✅ Official submission-form draft generated
✅ All character-limited fields are within official limits
 - project_abstract: 893/1000
 - rationale_impact_innovation: 2068/3500
 - design_implementation: 3531/7000
 - evaluation_sustainability: 2221/3500
 - twitter_summary: 102/140
 - fhir_use: 410/500
 - fhir_release_resources: 433/500
 - fhir_data_source_access: 431/500
 - other_information: 1022/1500
✅ Form draft: /content/drive/MyDrive/neurofhir-qc/submission/amia_2026/01_submission_form/AMIA_2026_SUBMISSION_FORM_DRAFT.md


In [10]:
# Cell 4 — Generate FHIR resource, CapabilityStatement, provenance, and measured-results artifacts

resource_rows = [
    {
        "resource": "Patient",
        "use": "Synthetic patient context",
        "demonstrated": True,
    },
    {
        "resource": "Condition",
        "use": "Synthetic neurological condition or research phenotype",
        "demonstrated": True,
    },
    {
        "resource": "ImagingStudy",
        "use": "Links public research MRI metadata to synthetic context",
        "demonstrated": True,
    },
    {
        "resource": "Observation",
        "use": "Prior reviewed and current AI-derived volume, QC components, status",
        "demonstrated": True,
    },
    {
        "resource": "DiagnosticReport",
        "use": "AI-assisted imaging-biomarker summary and reviewed conclusion",
        "demonstrated": True,
    },
    {
        "resource": "Device",
        "use": "Model, software, version, and checkpoint identity",
        "demonstrated": True,
    },
    {
        "resource": "Provenance",
        "use": "Algorithmic generation and human-review audit events",
        "demonstrated": True,
    },
    {
        "resource": "Task",
        "use": "Requested, on-hold, completed, and rejected review workflow",
        "demonstrated": True,
    },
    {
        "resource": "Practitioner",
        "use": "Explicitly synthetic research reviewer",
        "demonstrated": True,
    },
    {
        "resource": "Bundle",
        "use": "Self-contained deterministic FHIR transaction write-back",
        "demonstrated": True,
    },
    {
        "resource": "CapabilityStatement",
        "use": "Declares the demonstrated app-side R4 interactions",
        "demonstrated": True,
    },
]

resource_table_lines = [
    "# NeuroFHIR-QC FHIR R4 Resource List",
    "",
    "> This list reflects resources actually demonstrated in the executed prototype.",
    "",
    "| Resource | Demonstrated use |",
    "|---|---|",
]
for row in resource_rows:
    resource_table_lines.append(
        f"| `{row['resource']}` | {row['use']} |"
    )

resource_table_lines.extend(
    [
        "",
        "## FHIR technologies",
        "",
        "- FHIR R4 / 4.0.1",
        "- REST API",
        "- CapabilityStatement discovery",
        "- `$validate`",
        "- OperationOutcome",
        "- transaction Bundle",
        "- deterministic PUT",
        "- server write-back and direct read-back",
        "- deterministic URN fullUrl references within Bundles",
        "- HAPI FHIR public R4 sandbox",
        "",
        "## Not currently claimed",
        "",
        "- US Core profile conformance",
        "- SMART App Launch",
        "- CDS Hooks",
        "- Bulk FHIR",
        "- CQL",
        "- production EHR integration",
        "- hospital deployment",
    ]
)
RESOURCE_LIST_PATH.write_text(
    "\n".join(resource_table_lines).strip() + "\n",
    encoding="utf-8",
)

capability_resources = []
for row in resource_rows:
    resource_type = row["resource"]
    if resource_type == "CapabilityStatement":
        continue
    interactions = [
        {"code": "read"},
    ]
    if resource_type == "Bundle":
        interactions = [{"code": "create"}]
    else:
        interactions.extend(
            [
                {"code": "create"},
                {"code": "update"},
            ]
        )
    capability_resources.append(
        {
            "type": resource_type,
            "interaction": interactions,
        }
    )

app_capability_statement = {
    "resourceType": "CapabilityStatement",
    "id": "neurofhir-qc-app-capability",
    "meta": {
        "tag": [
            {
                "system": (
                    "https://neurofhir-qc.org/fhir/"
                    "CodeSystem/data-origin"
                ),
                "code": "research-prototype",
                "display": "Research prototype",
            }
        ]
    },
    "url": (
        "https://neurofhir-qc.org/fhir/"
        "CapabilityStatement/neurofhir-qc-app-capability"
    ),
    "version": str(project_config.get("version", "0.1.0")),
    "name": "NeuroFHIRQCAppCapability",
    "title": (
        "NeuroFHIR-QC Research Prototype "
        "Application CapabilityStatement"
    ),
    "status": "active",
    "experimental": True,
    "date": utc_now(),
    "publisher": SUBMITTER_NAME,
    "description": (
        "App-side capability statement for the executed "
        "NeuroFHIR-QC research MVP. It describes demonstrated "
        "FHIR R4 client interactions and is not a production "
        "server conformance claim."
    ),
    "kind": "capability",
    "implementation": {
        "description": (
            "Public research prototype repository and "
            "HAPI FHIR R4 sandbox workflow"
        ),
        "url": REPOSITORY_URL,
    },
    "fhirVersion": "4.0.1",
    "format": ["json"],
    "rest": [
        {
            "mode": "client",
            "documentation": (
                "Reads synthetic context; validates, creates, "
                "updates, writes transaction Bundles, and "
                "reads generated evidence back."
            ),
            "resource": capability_resources,
            "interaction": [
                {
                    "code": "transaction",
                }
            ],
        }
    ],
}
write_json(APP_CAPABILITY_PATH, app_capability_statement)

provenance_rows = [
    {
        "element": "Source image",
        "captured": True,
        "evidence": (
            "ImagingStudy, derivedFrom/entity references, "
            "and Notebook 04 source artifacts"
        ),
    },
    {
        "element": "Input biomarker or segmentation artifact",
        "captured": True,
        "evidence": (
            "Observation, Provenance entity, and mask artifact"
        ),
    },
    {
        "element": "Model name",
        "captured": True,
        "evidence": "Device and algorithmic Provenance",
    },
    {
        "element": "Model version",
        "captured": True,
        "evidence": "Device.version",
    },
    {
        "element": "Pipeline version",
        "captured": True,
        "evidence": "Device and notebook audit metadata",
    },
    {
        "element": "Timestamp",
        "captured": True,
        "evidence": (
            "Observation.issued, Provenance.recorded, "
            "Bundle.timestamp, audit timestamps"
        ),
    },
    {
        "element": "Human reviewer",
        "captured": True,
        "evidence": (
            "Synthetic Practitioner and review Provenance.agent"
        ),
    },
    {
        "element": "Review decision",
        "captured": True,
        "evidence": (
            "Task businessStatus/output and "
            "review Provenance.activity/reason"
        ),
    },
    {
        "element": "QC score and category",
        "captured": True,
        "evidence": "Observation components and Notebook 05 QC artifacts",
    },
]
with PROVENANCE_CHECKLIST_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(provenance_rows[0].keys()),
    )
    writer.writeheader()
    writer.writerows(provenance_rows)

provenance_completeness_rate = (
    sum(row["captured"] for row in provenance_rows)
    / len(provenance_rows)
)
if provenance_completeness_rate != 1.0:
    raise AssertionError(
        "Provenance completeness is not 100%."
    )

measured_rows = [
    {
        "domain": "Segmentation",
        "metric": "Public research MRI cases",
        "result": case_count,
        "denominator_or_unit": "cases",
        "claim_boundary": "Executable demonstration benchmark",
    },
    {
        "domain": "Segmentation",
        "metric": "Mean whole-tumor Dice",
        "result": round(mean_dice, 4),
        "denominator_or_unit": "0–1",
        "claim_boundary": "Not independent clinical validation",
    },
    {
        "domain": "Volumetry",
        "metric": "Mean absolute volume error",
        "result": round(mean_abs_volume_error_ml, 3),
        "denominator_or_unit": "mL",
        "claim_boundary": "Three demonstration cases",
    },
    {
        "domain": "Runtime",
        "metric": "Mean inference time",
        "result": round(mean_inference_seconds, 2),
        "denominator_or_unit": "seconds",
        "claim_boundary": "Archived notebook inference runtime",
    },
    {
        "domain": "Robustness",
        "metric": "Controlled perturbation types",
        "result": standard_perturbation_types,
        "denominator_or_unit": "types",
        "claim_boundary": "Engineering robustness audit",
    },
    {
        "domain": "Robustness",
        "metric": "Standard perturbation runs",
        "result": standard_perturbation_runs,
        "denominator_or_unit": "runs",
        "claim_boundary": "Three cases × four perturbations",
    },
    {
        "domain": "Robustness",
        "metric": "Severe challenge runs",
        "result": challenge_runs,
        "denominator_or_unit": "run",
        "claim_boundary": "Deliberate synthetic failure case",
    },
    {
        "domain": "QC safety",
        "metric": "Low-confidence detection",
        "result": "100%",
        "denominator_or_unit": "1/1 severe challenge",
        "claim_boundary": "Scripted demonstration behavior",
    },
    {
        "domain": "Longitudinal",
        "metric": "Scenario alignment",
        "result": "3/3",
        "denominator_or_unit": "cases",
        "claim_boundary": "Stable, progression, low-confidence",
    },
    {
        "domain": "FHIR",
        "metric": "Server validation targets passed",
        "result": f"{validation_pass_count}/{validation_targets}",
        "denominator_or_unit": "targets",
        "claim_boundary": "Public HAPI R4 sandbox",
    },
    {
        "domain": "FHIR",
        "metric": "Transaction Bundles succeeded",
        "result": f"{transaction_success_count}/{transaction_count}",
        "denominator_or_unit": "Bundles",
        "claim_boundary": "Deterministic PUT transactions",
    },
    {
        "domain": "FHIR",
        "metric": "Transaction entries succeeded",
        "result": f"{successful_entry_count}/{submitted_entry_count}",
        "denominator_or_unit": "entries",
        "claim_boundary": "Public HAPI R4 sandbox",
    },
    {
        "domain": "FHIR",
        "metric": "Critical-field preservation",
        "result": "100%",
        "denominator_or_unit": f"{readback_resource_count} resources",
        "claim_boundary": "Direct read-back comparison",
    },
    {
        "domain": "Workflow",
        "metric": "Review transition events",
        "result": review_transition_event_count,
        "denominator_or_unit": "events",
        "claim_boundary": "Synthetic research reviewer",
    },
    {
        "domain": "Workflow",
        "metric": "Accepted / correction-required / rejected",
        "result": (
            f"{accepted_case_count} / "
            f"{correction_required_case_count} / "
            f"{rejected_case_count}"
        ),
        "denominator_or_unit": "case events",
        "claim_boundary": "Scripted workflow mechanics",
    },
    {
        "domain": "Safety",
        "metric": "Low-confidence finalization block",
        "result": "100%",
        "denominator_or_unit": "unstable case",
        "claim_boundary": "No autonomous finalization",
    },
    {
        "domain": "Provenance",
        "metric": "Required elements captured",
        "result": f"{len(provenance_rows)}/{len(provenance_rows)}",
        "denominator_or_unit": "elements",
        "claim_boundary": "Executed artifact checklist",
    },
]

with MEASURED_RESULTS_CSV.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(measured_rows[0].keys()),
    )
    writer.writeheader()
    writer.writerows(measured_rows)

measured_md_lines = [
    "# NeuroFHIR-QC Measured Results",
    "",
    "| Domain | Metric | Result | Unit/denominator | Claim boundary |",
    "|---|---|---:|---|---|",
]
for row in measured_rows:
    measured_md_lines.append(
        f"| {row['domain']} | {row['metric']} | "
        f"{row['result']} | {row['denominator_or_unit']} | "
        f"{row['claim_boundary']} |"
    )
MEASURED_RESULTS_MD.write_text(
    "\n".join(measured_md_lines).strip() + "\n",
    encoding="utf-8",
)

print("=" * 112)
print("✅ App CapabilityStatement generated")
print(f"✅ Demonstrated FHIR resources: {len(resource_rows)}")
print(
    f"✅ Provenance completeness: "
    f"{len(provenance_rows)}/{len(provenance_rows)}"
)
print(f"✅ Measured result rows: {len(measured_rows)}")
print("=" * 112)

✅ App CapabilityStatement generated
✅ Demonstrated FHIR resources: 11
✅ Provenance completeness: 9/9
✅ Measured result rows: 17


In [11]:
# Cell 5 — Generate the eight-minute pitch, run of show, video storyboard, slides, and screenshot list

DEMO_SCRIPT_PATH.write_text(
    textwrap.dedent(
        f"""
        # NeuroFHIR-QC Eight-Minute Competition Presentation

        **Target length:** 7:40–7:55, leaving a few seconds of safety margin.
        **Central demonstration:** detect an unstable AI result, keep it non-final, route it for review, and preserve the full FHIR evidence trail.

        ## 0:00–0:40 — The problem

        Computational neuroimaging AI can produce a segmentation mask and a volume, but that output often remains isolated from patient context, source MRI, model version, longitudinal history, quality evidence, and human review. NeuroFHIR-QC addresses that missing informatics layer.

        **Say:**
        “NeuroFHIR-QC transforms AI-generated longitudinal brain-tumor or lesion measurements into quality-scored, provenance-aware, human-reviewed FHIR evidence.”

        ## 0:40–1:20 — What the prototype does

        Show the workflow graphic:

        `Synthetic FHIR context → public MRI → segmentation and volume → perturbation QC → longitudinal comparison → review Task → FHIR transaction and Provenance`

        State the boundary clearly: public de-identified research MRI, synthetic FHIR R4 records, and no clinical-deployment claim.

        ## 1:20–2:00 — Stable case

        Show the stable case:

        - prior reviewed volume: approximately 18.66 mL;
        - current AI-derived volume: approximately 19.19 mL;
        - change: approximately +2.82%;
        - QC: High confidence;
        - initial status: preliminary.

        Explain that a plausible result is still not automatically final.

        ## 2:00–2:40 — Progression case

        Show the progression case:

        - prior reviewed volume: 12.50 mL;
        - current AI-derived volume: approximately 20.46 mL;
        - change: approximately +63.71%;
        - QC: High confidence;
        - status remains preliminary until review.

        This demonstrates that the system can distinguish stable and meaningful increase while retaining human oversight.

        ## 2:40–4:05 — The strongest moment: unstable result

        Show the severe low-confidence challenge:

        - the numerical change is approximately −89.9%;
        - the QC score is approximately 0.339;
        - category: Manual review required;
        - the longitudinal interpretation is withheld;
        - autonomous finalization is blocked.

        **Say:**
        “The most important result is not that the model always succeeds. It is that the system recognizes instability, refuses to convert uncertainty into a final result, and preserves what happened.”

        Show the Task entering correction-required/on-hold. Then show that the original uncorrected output is rejected and its Observation and DiagnosticReport are marked entered-in-error.

        ## 4:05–5:10 — Human-review governance

        Show the review evidence:

        - stable accepted and finalized;
        - progression accepted and finalized;
        - low-confidence correction-required state preserved;
        - low-confidence original result rejected;
        - reviewer role, timestamp, reason, note, and decision stored;
        - four review Provenance events.

        State that the current reviewer is synthetic and demonstrates workflow mechanics rather than clinician usability.

        ## 5:10–6:20 — FHIR is central

        Show the resource graph and representative JSON.

        Resources demonstrated:

        - Patient;
        - Condition;
        - ImagingStudy;
        - Observation;
        - DiagnosticReport;
        - Device;
        - Provenance;
        - Task;
        - Practitioner;
        - Bundle.

        Explain:

        - FHIR R4 / 4.0.1;
        - preliminary and final status transitions;
        - transaction Bundles;
        - deterministic PUT;
        - same-Bundle URN references;
        - `$validate`;
        - OperationOutcome;
        - server write-back and direct read-back.

        ## 6:20–7:10 — Quantitative evidence

        Present only measured results:

        - mean whole-tumor Dice: {mean_dice:.4f};
        - mean absolute volume error: {mean_abs_volume_error_ml:.3f} mL;
        - four perturbation types, {standard_perturbation_runs} standard runs, one severe challenge;
        - longitudinal scenario alignment: 3/3;
        - FHIR validation: {validation_pass_count}/{validation_targets};
        - transactions: {transaction_success_count}/{transaction_count};
        - transaction entries: {successful_entry_count}/{submitted_entry_count};
        - critical-field preservation: 100%.

        ## 7:10–7:45 — Why it matters

        NeuroFHIR-QC is not another segmentation model and not merely a visualization. It is an interoperability and governance layer that converts computational neuro-AI outputs into traceable, quality-aware, reviewable, standards-based research evidence.

        ## 7:45–7:55 — Close

        **Closing line:**
        “NeuroFHIR-QC makes uncertainty visible, keeps AI evidence reviewable, and uses FHIR to preserve the complete path from source image to human decision.”
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

RUN_OF_SHOW_PATH.write_text(
    textwrap.dedent(
        """
        # Live Demo Run of Show

        ## Pre-demo checks

        1. Open the app or prerecorded demonstration before the session.
        2. Confirm the selected synthetic patient is visible.
        3. Confirm the stable, progression, and low-confidence cases load.
        4. Confirm FHIR JSON, validation, transaction, and Provenance views are available.
        5. Keep a local screen-recording backup ready.
        6. Do not depend on a live public HAPI write during the eight-minute stage presentation.
        7. Use archived successful validation and transaction evidence for deterministic presentation.

        ## Demo sequence

        | Time | Action | Evidence shown |
        |---|---|---|
        | 0:00–0:40 | State problem and pitch | workflow overview |
        | 0:40–1:20 | Open synthetic patient | Patient, Condition, ImagingStudy |
        | 1:20–2:00 | Stable case | volume, +2.82%, QC High confidence |
        | 2:00–2:40 | Progression case | volume, +63.71%, QC High confidence |
        | 2:40–4:05 | Low-confidence case | QC 0.339, interpretation withheld |
        | 4:05–5:10 | Review transitions | accept, correction-required, reject |
        | 5:10–6:20 | FHIR audit view | JSON, Bundle, validation, graph, Provenance |
        | 6:20–7:10 | Measured results | evaluation table |
        | 7:10–7:55 | impact and close | final workflow graphic |

        ## Failure-safe presentation rule

        The stage presentation must remain complete even if the public FHIR sandbox is temporarily unavailable. Use archived OperationOutcome, transaction-response, read-back, and reference-graph evidence created by the executed notebooks.
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

VIDEO_STORYBOARD_PATH.write_text(
    textwrap.dedent(
        """
        # Demonstration Video Storyboard

        **Recommended length:** 4–6 minutes for reviewer access; create a separate eight-minute finalist presentation.

        1. **Opening title and problem — 20 seconds**
           - NeuroFHIR-QC title
           - one-sentence pitch
           - public MRI + synthetic FHIR boundary

        2. **Patient and imaging context — 35 seconds**
           - Patient
           - Condition
           - prior and current ImagingStudy
           - prior reviewed Observation

        3. **Stable and progression cases — 60 seconds**
           - volume cards
           - longitudinal change
           - QC category
           - preliminary status

        4. **Low-confidence failure — 80 seconds**
           - instability evidence
           - QC score/category
           - interpretation withheld
           - finalization blocked

        5. **Human review — 60 seconds**
           - accept stable
           - accept progression
           - correction-required low confidence
           - reject uncorrected result
           - show Task and Provenance transitions

        6. **FHIR evidence — 60 seconds**
           - resource graph
           - representative Observation
           - transaction Bundle
           - OperationOutcome
           - read-back evidence

        7. **Measured evaluation and limitations — 45 seconds**
           - three-case Dice and volume error
           - robustness runs
           - validation/transaction/read-back rates
           - no clinical or human-usability claim

        8. **Closing — 20 seconds**
           - “Make uncertainty visible and reviewable in FHIR.”
           - repository and contact
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

SLIDE_OUTLINE_PATH.write_text(
    textwrap.dedent(
        f"""
        # Eight-Minute Slide Deck Outline

        ## Slide 1 — NeuroFHIR-QC

        - Trustworthy Longitudinal Neuroimaging AI in FHIR
        - one-sentence pitch
        - public research MRI + synthetic FHIR R4

        ## Slide 2 — The missing informatics layer

        - fragmented MRI, masks, notebooks, QC, review notes
        - missing patient linkage, versioning, status, and provenance
        - workflow diagram

        ## Slide 3 — Closed MVP workflow

        - source context
        - segmentation and volumetry
        - perturbation QC
        - longitudinal comparison
        - review
        - FHIR validation/write-back/read-back

        ## Slide 4 — Three demonstration behaviors

        - stable: +2.82%
        - progression: +63.71%
        - low confidence: −89.9%, interpretation withheld
        - all AI results begin preliminary

        ## Slide 5 — Responsible failure handling

        - QC score approximately 0.339
        - Manual review required
        - correction-required/on-hold
        - rejected/entered-in-error
        - strongest competition moment

        ## Slide 6 — FHIR evidence graph

        - Patient, Condition, ImagingStudy
        - Observation, DiagnosticReport, Device
        - Task, Provenance, Practitioner, Bundle
        - `$validate`, transaction, read-back

        ## Slide 7 — Human-review state machine

        - accepted → final/completed
        - correction-required → preliminary/on-hold
        - rejected → entered-in-error/rejected
        - four review Provenance events

        ## Slide 8 — Measured evaluation

        - mean Dice {mean_dice:.4f}
        - mean absolute volume error {mean_abs_volume_error_ml:.3f} mL
        - {standard_perturbation_runs} standard perturbation runs
        - 3/3 longitudinal alignment
        - {validation_pass_count}/{validation_targets} validation targets
        - {successful_entry_count}/{submitted_entry_count} transaction entries
        - 100% read-back field preservation

        ## Slide 9 — What is proven and what is not

        Proven:
        - executable technical workflow
        - FHIR conformance in tested sandbox cases
        - responsible low-confidence handling
        - reproducible evidence package

        Not claimed:
        - clinical validation
        - hospital deployment
        - patient impact
        - real clinician review
        - human usability score

        ## Slide 10 — Close

        “NeuroFHIR-QC makes uncertainty visible, keeps AI evidence reviewable, and preserves the full path from source image to human decision in FHIR.”
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

SCREENSHOT_LIST_PATH.write_text(
    textwrap.dedent(
        """
        # Competition Screenshot Shot List

        No screenshot is marked complete unless it is captured from the real executed prototype or its archived evidence.

        | ID | Required image | Must show | Status |
        |---|---|---|---|
        | S01 | Landing/workflow | problem, closed workflow, data boundary | NOT CAPTURED |
        | S02 | Synthetic patient context | Patient, Condition, prior/current ImagingStudy | NOT CAPTURED |
        | S03 | Stable biomarker | 18.66→19.19 mL, +2.82%, High confidence | NOT CAPTURED |
        | S04 | Progression biomarker | 12.50→20.46 mL, +63.71%, High confidence | NOT CAPTURED |
        | S05 | Low-confidence QC | score ~0.339, Manual review required | NOT CAPTURED |
        | S06 | Interpretation withheld | unstable value retained but interpretation withheld | NOT CAPTURED |
        | S07 | Correction-required | Task on-hold and review reason/note | NOT CAPTURED |
        | S08 | Rejected result | entered-in-error and rejected Task | NOT CAPTURED |
        | S09 | FHIR resource graph | linked source and generated resources | NOT CAPTURED |
        | S10 | FHIR validation | successful OperationOutcome/validation summary | NOT CAPTURED |
        | S11 | Transaction/read-back | successful Bundle response and integrity | NOT CAPTURED |
        | S12 | Evaluation dashboard | Dice, robustness, FHIR, workflow metrics | NOT CAPTURED |
        | S13 | Repository/reproducibility | notebooks, audits, docs, package | NOT CAPTURED |
        | S14 | Promotional composite | application, MRI overlay, FHIR graph | NOT CAPTURED |

        ## Screenshot rules

        - Use synthetic patient identifiers only.
        - Do not show personal browser accounts, Drive paths, tokens, or private email.
        - Do not show unsupported “clinically validated” or “deployed” labels.
        - Make FHIR visible; do not hide it behind only a dashboard.
        - Capture the low-confidence path as the hero image.
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

print("=" * 112)
print("✅ Eight-minute presentation script generated")
print("✅ Live-demo run of show generated")
print("✅ Demo-video storyboard generated")
print("✅ Ten-slide outline generated")
print("✅ Fourteen-shot screenshot checklist generated")
print("=" * 112)

✅ Eight-minute presentation script generated
✅ Live-demo run of show generated
✅ Demo-video storyboard generated
✅ Ten-slide outline generated
✅ Fourteen-shot screenshot checklist generated


In [12]:
# Cell 6 — Copy representative executed evidence into the competition package

copy_candidates = {
    "project_config.json": PROJECT_CONFIG_PATH,
    "notebook_manifest.json": NOTEBOOK_MANIFEST_PATH,
    "notebook_09_audit.json": NB09_AUDIT_PATH,
    "competition_scorecard.json": NB09_SCORECARD_PATH,
    "segmentation_evaluation.json": NB09_SEGMENTATION_PATH,
    "robustness_qc_evaluation.json": NB09_ROBUSTNESS_PATH,
    "longitudinal_evaluation.json": NB09_LONGITUDINAL_PATH,
    "fhir_interoperability_evaluation.json":
        NB09_INTEROPERABILITY_PATH,
    "workflow_safety_evaluation.json": NB09_WORKFLOW_PATH,
    "timing_evaluation.json": NB09_TIMING_PATH,
    "usability_readiness.json": NB09_USABILITY_PATH,
    "reproducibility_evaluation.json":
        NB09_REPRODUCIBILITY_PATH,
    "integrated_evaluation_report.md":
        DOC_ROOT / "NOTEBOOK_09_INTEGRATED_EVALUATION.md",
    "verified_competition_claims.md":
        DOC_ROOT / "VERIFIED_COMPETITION_CLAIMS.md",
    "evaluation_limitations.md":
        DOC_ROOT / "EVALUATION_LIMITATIONS.md",
    "human_review_workflow.md":
        DOC_ROOT / "HUMAN_REVIEW_WORKFLOW.md",
    "usability_evaluation_protocol.md":
        DOC_ROOT / "USABILITY_EVALUATION_PROTOCOL.md",
    "notebook_07_validation_report.json":
        NB07_ROOT / "server_validation_report.json",
    "notebook_07_transaction_report.json":
        NB07_ROOT / "transaction_writeback_report.json",
    "notebook_07_readback_report.json":
        NB07_ROOT / "readback_integrity_report.json",
    "notebook_07_reference_graph.json":
        NB07_ROOT / "fhir_reference_graph.json",
    "notebook_08_validation_report.json":
        NB08_ROOT / "server_validation_report.json",
    "notebook_08_transaction_report.json":
        NB08_ROOT / "transaction_report.json",
    "notebook_08_readback_report.json":
        NB08_ROOT / "readback_integrity_report.json",
    "notebook_08_transition_report.json":
        NB08_ROOT / "review_transition_report.json",
    "notebook_07_master_evidence_bundle.json":
        NB07_SUBMISSION_ROOT / "master_evidence_collection_bundle.json",
    "notebook_08_master_review_evidence_bundle.json":
        NB08_SUBMISSION_ROOT / "master_review_evidence_bundle.json",
}

required_copy_names = {
    "project_config.json",
    "notebook_manifest.json",
    "notebook_09_audit.json",
    "competition_scorecard.json",
    "segmentation_evaluation.json",
    "robustness_qc_evaluation.json",
    "longitudinal_evaluation.json",
    "fhir_interoperability_evaluation.json",
    "workflow_safety_evaluation.json",
    "notebook_07_validation_report.json",
    "notebook_07_transaction_report.json",
    "notebook_07_readback_report.json",
    "notebook_08_validation_report.json",
    "notebook_08_transaction_report.json",
    "notebook_08_readback_report.json",
    "notebook_08_transition_report.json",
    "notebook_07_master_evidence_bundle.json",
    "notebook_08_master_review_evidence_bundle.json",
}

copy_rows = []
for destination_name, source_path in copy_candidates.items():
    exists = (
        source_path.exists()
        and source_path.stat().st_size > 0
    )
    if not exists and destination_name in required_copy_names:
        raise FileNotFoundError(
            f"Required evidence file missing: {source_path}"
        )
    if not exists:
        copy_rows.append(
            {
                "destination_name": destination_name,
                "source_relative_path": (
                    source_path.relative_to(PROJECT_ROOT).as_posix()
                    if source_path.is_relative_to(PROJECT_ROOT)
                    else str(source_path)
                ),
                "copied": False,
                "reason": "optional source not found",
            }
        )
        continue

    destination_root = (
        FHIR_ROOT
        if "bundle" in destination_name
        or "validation" in destination_name
        or "transaction" in destination_name
        or "readback" in destination_name
        or "reference_graph" in destination_name
        else EVIDENCE_ROOT
    )
    destination_path = destination_root / destination_name
    shutil.copy2(source_path, destination_path)
    if sha256_file(source_path) != sha256_file(
        destination_path
    ):
        raise AssertionError(
            f"Copied evidence checksum mismatch: {source_path}"
        )
    copy_rows.append(
        {
            "destination_name": destination_name,
            "source_relative_path": source_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
            "destination_relative_path":
                destination_path.relative_to(
                    PACKAGE_ROOT
                ).as_posix(),
            "copied": True,
            "size_bytes": destination_path.stat().st_size,
            "sha256": sha256_file(destination_path),
        }
    )

plot_source_root = NB09_ROOT / "plots"
plot_copy_rows = []
if plot_source_root.exists():
    for source_plot in sorted(
        plot_source_root.glob("*.png")
    ):
        destination_plot = VISUAL_ROOT / source_plot.name
        shutil.copy2(source_plot, destination_plot)
        plot_copy_rows.append(
            {
                "source_relative_path":
                    source_plot.relative_to(
                        PROJECT_ROOT
                    ).as_posix(),
                "destination_relative_path":
                    destination_plot.relative_to(
                        PACKAGE_ROOT
                    ).as_posix(),
                "size_bytes":
                    destination_plot.stat().st_size,
                "sha256": sha256_file(destination_plot),
            }
        )

with (
    EVIDENCE_ROOT / "copied_evidence_inventory.csv"
).open("w", newline="", encoding="utf-8") as handle:
    fieldnames = sorted(
        {
            key
            for row in copy_rows
            for key in row.keys()
        }
    )
    writer = csv.DictWriter(
        handle,
        fieldnames=fieldnames,
    )
    writer.writeheader()
    writer.writerows(copy_rows)

write_json(
    VISUAL_ROOT / "visual_inventory.json",
    {
        "generated_utc": utc_now(),
        "copied_plot_count": len(plot_copy_rows),
        "plots": plot_copy_rows,
        "competition_screenshots_captured": False,
        "screenshot_checklist": (
            SCREENSHOT_LIST_PATH.relative_to(
                PACKAGE_ROOT
            ).as_posix()
        ),
    },
)

print("=" * 112)
print(
    f"✅ Evidence files copied: "
    f"{sum(row['copied'] for row in copy_rows)}"
)
print(f"✅ Evaluation plots copied: {len(plot_copy_rows)}")
print("✅ Raw MRI, masks, model checkpoints, and PHI were not packaged")
print("=" * 112)

✅ Evidence files copied: 27
✅ Evaluation plots copied: 10
✅ Raw MRI, masks, model checkpoints, and PHI were not packaged


In [13]:
# Cell 7 — Evaluate category eligibility and create the final submission checklist

valid_categories = {
    "Academic",
    "Industry",
    "Student",
}
category_is_valid = SUBMISSION_CATEGORY in valid_categories

if SUBMISSION_CATEGORY == "Student":
    category_eligibility_satisfied = (
        STUDENT_ADVISOR_ATTESTATION_AVAILABLE
    )
    category_reason = (
        "Student category selected. Official eligibility requires "
        "a minimum viable prototype and a signed primary-advisor "
        "attestation. The technical MVP is present; attestation "
        f"availability={STUDENT_ADVISOR_ATTESTATION_AVAILABLE}."
    )
elif SUBMISSION_CATEGORY in {"Academic", "Industry"}:
    category_eligibility_satisfied = (
        REAL_WORLD_PRACTICE_EVIDENCE_AVAILABLE
    )
    category_reason = (
        f"{SUBMISSION_CATEGORY} category selected. Official "
        "eligibility requires current use in real-world practice "
        "for a non-student submission. Real-world-use evidence "
        f"availability={REAL_WORLD_PRACTICE_EVIDENCE_AVAILABLE}."
    )
else:
    category_eligibility_satisfied = False
    category_reason = (
        "Category is unresolved. Select Academic, Industry, or "
        "Student only after confirming official eligibility."
    )

branding_status = {
    "logo": LOGO_PATH.exists(),
    "headshot": HEADSHOT_PATH.exists(),
    "promotional_photo": PROMOTIONAL_PHOTO_PATH.exists(),
}
branding_complete = all(branding_status.values())

administrative_status = {
    "category_selected": category_is_valid,
    "category_eligibility_supported":
        category_eligibility_satisfied,
    "current_affiliation_verified": (
        bool(AFFILIATION)
        and "VERIFY" not in AFFILIATION.upper()
    ),
    "conception_date_verified": bool(CONCEPTION_DATE),
    "implementation_date_available":
        bool(IMPLEMENTATION_DATE),
    "app_or_demo_url_available":
        bool(APP_OR_DEMO_URL),
    "repository_url_available": bool(REPOSITORY_URL),
    "logo_available": branding_status["logo"],
    "headshot_available": branding_status["headshot"],
    "promotional_photo_available":
        branding_status["promotional_photo"],
    "smart_gallery_agreement_resolved": (
        SMART_GALLERY_AGREEMENT.lower()
        in {"yes", "no"}
    ),
}

technical_package_status = {
    "submission_form_draft": FORM_DRAFT_MD.exists(),
    "character_limit_audit":
        CHARACTER_AUDIT_CSV.exists(),
    "fhir_resource_list": RESOURCE_LIST_PATH.exists(),
    "app_capability_statement":
        APP_CAPABILITY_PATH.exists(),
    "measured_results": MEASURED_RESULTS_CSV.exists(),
    "provenance_checklist":
        PROVENANCE_CHECKLIST_CSV.exists(),
    "eight_minute_script": DEMO_SCRIPT_PATH.exists(),
    "video_storyboard": VIDEO_STORYBOARD_PATH.exists(),
    "slide_outline": SLIDE_OUTLINE_PATH.exists(),
    "screenshot_list": SCREENSHOT_LIST_PATH.exists(),
    "verified_claims": PORTAL_CLAIMS_PATH.exists(),
}

technical_package_complete = all(
    technical_package_status.values()
)
portal_ready = (
    technical_package_complete
    and all(administrative_status.values())
)

CATEGORY_DECISION_PATH.write_text(
    textwrap.dedent(
        f"""
        # Category Eligibility Decision

        **Selected category:** {SUBMISSION_CATEGORY}
        **Category valid:** {category_is_valid}
        **Eligibility support available:** {category_eligibility_satisfied}
        **Portal-ready eligibility gate:** {portal_ready}

        ## Official rule applied

        - Non-student submissions must currently be used in real-world practice.
        - Student submissions must have a minimum viable prototype.
        - Student submissions require a signed primary-advisor attestation.

        ## Current assessment

        {category_reason}

        ## Required decision

        Do not submit as Academic or Industry without truthful evidence of
        current real-world use. Do not submit as Student without confirming
        eligibility and uploading the signed advisor attestation.

        The repository's technical implementation is a minimum viable
        research prototype, but technical completion does not by itself
        resolve category eligibility.
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

STUDENT_ATTESTATION_TEMPLATE_PATH.write_text(
    textwrap.dedent(
        f"""
        # Student Primary-Advisor Attestation Template

        **Competition:** 2026 AMIA/HL7 FHIR App Competition
        **App:** {title}
        **Student:** {SUBMITTER_NAME}

        To the AMIA/HL7 FHIR App Competition Review Committee:

        I attest to the following:

        1. **Training program name and address**
           [INSERT PROGRAM NAME]
           [INSERT PROGRAM ADDRESS]

        2. **Primary advisor**
           Name: [INSERT ADVISOR NAME]
           Title: [INSERT TITLE]
           Institution: [INSERT INSTITUTION]
           Contact: [INSERT CONTACT]

        3. **Co-authors and contributions**
           [LIST EVERY CO-AUTHOR AND DESCRIBE EACH CONTRIBUTION]

        4. **Student contribution**
           [DESCRIBE THE STUDENT'S SPECIFIC CONTRIBUTION TO DESIGN,
           IMPLEMENTATION, TESTING, FHIR MAPPING, EVALUATION, AND
           COMPETITION MATERIALS]

        I confirm that the statements above accurately represent the
        student's contribution to the development of the application.

        Advisor signature: ____________________
        Date: ____________________

        Advisor printed name: ____________________
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

NONSTUDENT_EVIDENCE_TEMPLATE_PATH.write_text(
    textwrap.dedent(
        """
        # Non-Student Real-World-Use Evidence Template

        Use this only for an Academic or Industry submission.

        ## Required truthful evidence

        - organization or setting where the app is currently used;
        - date real-world use began;
        - intended operational users and their roles;
        - number of active users and reporting period;
        - number of patients or records affected, if applicable;
        - production or operational URL/environment;
        - deployment owner and support contact;
        - evidence of routine use, such as usage logs, operational report,
          implementation letter, or documented workflow integration;
        - privacy, security, and governance context;
        - clear separation between real-world use and research testing.

        ## Current NeuroFHIR-QC status

        The executed repository currently supports a research MVP with
        public de-identified imaging and synthetic FHIR R4 context. Do not
        complete this template with invented users, patients, or deployment.
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

checklist_rows = [
    {
        "item": "Technical notebook evidence 04–09 completed",
        "required": True,
        "complete": True,
        "next_action": "None",
    },
    {
        "item": "Official form text within character limits",
        "required": True,
        "complete": True,
        "next_action": "Final human review before portal paste",
    },
    {
        "item": "FHIR CapabilityStatement or resource list",
        "required": True,
        "complete": True,
        "next_action": "Upload one or both",
    },
    {
        "item": "Repository URL",
        "required": True,
        "complete": bool(REPOSITORY_URL),
        "next_action": "Confirm public repository and README",
    },
    {
        "item": "Working app URL or reviewer-accessible video",
        "required": True,
        "complete": bool(APP_OR_DEMO_URL),
        "next_action": "Deploy the MVP UI or record/upload the demo",
    },
    {
        "item": "Submission category selected",
        "required": True,
        "complete": category_is_valid,
        "next_action": "Resolve Academic, Industry, or Student",
    },
    {
        "item": "Category eligibility evidence",
        "required": True,
        "complete": category_eligibility_satisfied,
        "next_action": (
            "Obtain advisor attestation for Student or truthful "
            "real-world-use evidence for non-student"
        ),
    },
    {
        "item": "Current affiliation verified",
        "required": True,
        "complete": administrative_status[
            "current_affiliation_verified"
        ],
        "next_action": "Enter current affiliation exactly",
    },
    {
        "item": "Conception date verified",
        "required": True,
        "complete": bool(CONCEPTION_DATE),
        "next_action": "Enter truthful conception month/date",
    },
    {
        "item": "Implementation date verified",
        "required": True,
        "complete": bool(IMPLEMENTATION_DATE),
        "next_action": "Confirm implementation date",
    },
    {
        "item": "Logo",
        "required": True,
        "complete": branding_status["logo"],
        "next_action": "Create and save logo image",
    },
    {
        "item": "Headshot",
        "required": True,
        "complete": branding_status["headshot"],
        "next_action": "Add approved professional headshot",
    },
    {
        "item": "Promotional image",
        "required": True,
        "complete": branding_status[
            "promotional_photo"
        ],
        "next_action": "Create app/MRI/FHIR promotional composite",
    },
    {
        "item": "Competition screenshots",
        "required": False,
        "complete": False,
        "next_action": "Capture the 14-shot list from the real prototype",
    },
    {
        "item": "SMART App Gallery agreement resolved",
        "required": True,
        "complete": administrative_status[
            "smart_gallery_agreement_resolved"
        ],
        "next_action": "Select yes or no in the portal",
    },
    {
        "item": "Human usability study",
        "required": False,
        "complete": bool(
            usability.get(
                "human_usability_study_performed",
                False,
            )
        ),
        "next_action": (
            "Run 5–10 honest evaluators if feasible; do not fabricate"
        ),
    },
    {
        "item": "Interactive UI task time and click count",
        "required": False,
        "complete": False,
        "next_action": "Measure after the final UI is available",
    },
]

required_incomplete = [
    row
    for row in checklist_rows
    if row["required"] and not row["complete"]
]

checklist_lines = [
    "# Final AMIA 2026 Submission Checklist",
    "",
    f"**Official deadline:** {OFFICIAL_DEADLINE}  ",
    f"**Technical package complete:** {technical_package_complete}  ",
    f"**Portal ready:** {portal_ready}  ",
    "",
    "| Item | Required | Complete | Next action |",
    "|---|---:|---:|---|",
]
for row in checklist_rows:
    checklist_lines.append(
        f"| {row['item']} | {row['required']} | "
        f"{row['complete']} | {row['next_action']} |"
    )
FINAL_CHECKLIST_PATH.write_text(
    "\n".join(checklist_lines).strip() + "\n",
    encoding="utf-8",
)

print("=" * 112)
print("✅ Category eligibility decision generated")
print(f"📋 Selected category: {SUBMISSION_CATEGORY}")
print(
    f"📋 Category eligibility supported: "
    f"{category_eligibility_satisfied}"
)
print(f"✅ Technical artifact package complete: {technical_package_complete}")
print(f"⚠️ Required portal items incomplete: {len(required_incomplete)}")
for row in required_incomplete:
    print(f" - {row['item']}: {row['next_action']}")
print(f"📦 Portal ready: {portal_ready}")
print("=" * 112)

✅ Category eligibility decision generated
📋 Selected category: Unresolved
📋 Category eligibility supported: False
✅ Technical artifact package complete: True
⚠️ Required portal items incomplete: 9
 - Working app URL or reviewer-accessible video: Deploy the MVP UI or record/upload the demo
 - Submission category selected: Resolve Academic, Industry, or Student
 - Category eligibility evidence: Obtain advisor attestation for Student or truthful real-world-use evidence for non-student
 - Current affiliation verified: Enter current affiliation exactly
 - Conception date verified: Enter truthful conception month/date
 - Logo: Create and save logo image
 - Headshot: Add approved professional headshot
 - Promotional image: Create app/MRI/FHIR promotional composite
 - SMART App Gallery agreement resolved: Select yes or no in the portal
📦 Portal ready: False


In [14]:
# Cell 8 — Create the package README and reviewer-facing HTML summary

PACKAGE_README_PATH.write_text(
    textwrap.dedent(
        f"""
        # NeuroFHIR-QC AMIA 2026 Submission Package

        ## Project

        **Title:** {title}
        **Pitch:** {competition_evidence['one_sentence_pitch']}
        **Repository:** {REPOSITORY_URL}
        **App/demo URL:** {APP_OR_DEMO_URL or 'NOT YET PROVIDED'}
        **Category:** {SUBMISSION_CATEGORY}
        **Portal ready:** {portal_ready}

        ## Official schedule

        - Submission deadline: {OFFICIAL_DEADLINE}
        - Finalist presentation: {OFFICIAL_PRESENTATION_DATE},
          {OFFICIAL_PRESENTATION_WINDOW}
        - Presentation length: {OFFICIAL_PRESENTATION_LENGTH}
        - Official rules: {OFFICIAL_RULES_URL}

        ## Package structure

        - `01_submission_form/` — portal-ready field draft, character audit,
          and verified claims.
        - `02_fhir/` — app CapabilityStatement, demonstrated resource list,
          representative Bundles, validation, transaction, read-back, and
          reference evidence.
        - `03_evidence/` — measured evaluation, audits, source-of-truth JSON,
          provenance checklist, and limitations.
        - `04_demo/` — eight-minute script, live run of show, slide outline,
          video storyboard, and screenshot list.
        - `05_visuals/` — copied evaluation plots. Competition screenshots
          must still be captured from the real prototype.
        - `06_eligibility/` — category decision, student attestation template,
          and non-student real-world-use evidence template.
        - `07_release/` — final checklist, artifact manifest, and SHA-256
          inventory.

        ## Measured execution highlights

        - Three public research MRI demonstration cases
        - Mean whole-tumor Dice: {mean_dice:.4f}
        - Mean absolute volume error: {mean_abs_volume_error_ml:.3f} mL
        - Four perturbation types, {standard_perturbation_runs} standard
          runs, and one severe challenge
        - Longitudinal scenario alignment: 3/3
        - FHIR validation: {validation_pass_count}/{validation_targets}
        - Transactions: {transaction_success_count}/{transaction_count}
        - Transaction entries: {successful_entry_count}/{submitted_entry_count}
        - Critical-field read-back preservation: 100%
        - Two accepted cases, one correction-required intermediate state,
          and one rejected unstable result
        - Four review Provenance events

        ## Claim boundary

        This is a research MVP using public de-identified imaging and
        synthetic FHIR R4 context. It does not establish clinical validity,
        production deployment, real clinician agreement, human usability,
        paying customers, patient impact, or non-student real-world use.

        ## Most important unresolved items

        {chr(10).join(f'- {row["item"]}: {row["next_action"]}' for row in required_incomplete) if required_incomplete else '- None'}

        ## Final packaging rule

        Upload or publish only after a human verifies every portal field,
        confirms eligibility, reviews the selected FHIR artifact, and
        confirms that branding and media contain no private information.
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

open_gap_items = "".join(
    (
        "<li><strong>"
        + html.escape(row["domain"])
        + ":</strong> "
        + html.escape(row["remaining_action"])
        + "</li>"
    )
    for row in competition_evidence[
        "open_evaluation_gaps"
    ]
)
required_gap_items = "".join(
    (
        "<li><strong>"
        + html.escape(row["item"])
        + ":</strong> "
        + html.escape(row["next_action"])
        + "</li>"
    )
    for row in required_incomplete
)

html_summary = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{html.escape(title)}</title>
<style>
body {{
  font-family: Arial, Helvetica, sans-serif;
  margin: 0;
  background: #f4f7fb;
  color: #172033;
}}
header {{
  background: linear-gradient(135deg, #14213d, #254b75);
  color: white;
  padding: 48px 24px;
}}
main {{
  max-width: 1050px;
  margin: 0 auto;
  padding: 24px;
}}
.card {{
  background: white;
  border-radius: 14px;
  padding: 22px;
  margin: 16px 0;
  box-shadow: 0 6px 22px rgba(20,33,61,.08);
}}
.metric-grid {{
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(190px, 1fr));
  gap: 14px;
}}
.metric {{
  background: #eef4fb;
  border-radius: 10px;
  padding: 16px;
}}
.metric b {{
  display: block;
  font-size: 1.65rem;
  margin-top: 5px;
}}
.good {{ color: #16794f; }}
.warn {{ color: #9a5b00; }}
small {{ color: #526078; }}
code {{
  background: #eef1f6;
  padding: 2px 5px;
  border-radius: 4px;
}}
</style>
</head>
<body>
<header>
<h1>{html.escape(title)}</h1>
<p>{html.escape(competition_evidence["one_sentence_pitch"])}</p>
<p><strong>Research MVP:</strong> public de-identified MRI + synthetic FHIR R4</p>
</header>
<main>
<section class="card">
<h2>Measured evidence</h2>
<div class="metric-grid">
<div class="metric">Mean Dice<b>{mean_dice:.4f}</b></div>
<div class="metric">Volume error<b>{mean_abs_volume_error_ml:.3f} mL</b></div>
<div class="metric">Longitudinal alignment<b>3/3</b></div>
<div class="metric">FHIR validation<b>{validation_pass_count}/{validation_targets}</b></div>
<div class="metric">Transactions<b>{transaction_success_count}/{transaction_count}</b></div>
<div class="metric">Entries<b>{successful_entry_count}/{submitted_entry_count}</b></div>
<div class="metric">Read-back preservation<b>100%</b></div>
<div class="metric">Review Provenance<b>{review_provenance_event_count}</b></div>
</div>
</section>
<section class="card">
<h2>Strongest demonstration</h2>
<p>An unstable segmentation is detected, remains non-final, has its
longitudinal interpretation withheld, enters a correction-required state,
and is rejected with complete FHIR Task and Provenance evidence.</p>
</section>
<section class="card">
<h2>FHIR R4 evidence</h2>
<p>Patient, Condition, ImagingStudy, Observation, DiagnosticReport,
Device, Provenance, Task, Practitioner, Bundle, and CapabilityStatement.</p>
<p><strong>Techniques:</strong> REST, <code>$validate</code>,
OperationOutcome, transaction Bundle, deterministic PUT, URN fullUrl,
write-back, direct read-back, and reference-integrity checks.</p>
</section>
<section class="card">
<h2>Submission readiness</h2>
<p class="{'good' if portal_ready else 'warn'}">
<strong>Portal ready: {portal_ready}</strong>
</p>
<ul>{required_gap_items or '<li>No required gaps.</li>'}</ul>
</section>
<section class="card">
<h2>Evaluation gaps retained honestly</h2>
<ul>{open_gap_items or '<li>No open evaluation gaps.</li>'}</ul>
</section>
<section class="card">
<h2>Claim boundary</h2>
<p>No clinical validation, hospital deployment, real clinician review,
paying customers, patient impact, or human usability score is claimed.</p>
<small>Generated {html.escape(utc_now())}</small>
</section>
</main>
</body>
</html>
"""
HTML_SUMMARY_PATH.write_text(
    html_summary,
    encoding="utf-8",
)

print("=" * 112)
print("✅ Submission-package README generated")
print("✅ Reviewer-facing HTML summary generated")
print(f"📄 Open locally after download: {HTML_SUMMARY_PATH}")
print("=" * 112)

✅ Submission-package README generated
✅ Reviewer-facing HTML summary generated
📄 Open locally after download: /content/drive/MyDrive/neurofhir-qc/submission/amia_2026/index.html


In [15]:
# Cell 9 — Build the artifact manifest, SHA-256 inventory, and ZIP archive

disallowed_suffixes = {
    ".nii",
    ".nii.gz",
    ".dcm",
    ".pt",
    ".pth",
    ".ckpt",
}
disallowed_names = {
    ".env",
    "credentials.json",
    "token.json",
    "secrets.json",
}

def is_disallowed(path: Path) -> bool:
    lower_name = path.name.lower()
    if lower_name in disallowed_names:
        return True
    if lower_name.endswith(".nii.gz"):
        return True
    if path.suffix.lower() in disallowed_suffixes:
        return True
    if ".ipynb_checkpoints" in path.parts:
        return True
    return False

pre_manifest_files = [
    path
    for path in PACKAGE_ROOT.rglob("*")
    if path.is_file()
    and path not in {
        ARTIFACT_MANIFEST_PATH,
        CHECKSUM_PATH,
    }
]

disallowed_files = [
    str(path)
    for path in pre_manifest_files
    if is_disallowed(path)
]
if disallowed_files:
    raise AssertionError(
        "Disallowed files found in competition package:\n"
        + "\n".join(f" - {path}" for path in disallowed_files)
    )

artifact_rows = [
    {
        "relative_path": path.relative_to(
            PACKAGE_ROOT
        ).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in sorted(pre_manifest_files)
]

artifact_manifest = {
    "project_name": "NeuroFHIR-QC",
    "package": "AMIA 2026 FHIR App Competition",
    "generated_utc": utc_now(),
    "official_deadline": OFFICIAL_DEADLINE,
    "official_rules_url": OFFICIAL_RULES_URL,
    "selected_category": SUBMISSION_CATEGORY,
    "category_eligibility_supported":
        category_eligibility_satisfied,
    "technical_package_complete":
        technical_package_complete,
    "portal_ready": portal_ready,
    "file_count_before_release_metadata":
        len(artifact_rows),
    "total_size_bytes_before_release_metadata":
        sum(row["size_bytes"] for row in artifact_rows),
    "data_boundary": {
        "public_deidentified_research_imaging": True,
        "synthetic_fhir": True,
        "real_patient_data": False,
        "raw_imaging_in_package": False,
        "model_checkpoint_in_package": False,
        "credentials_in_package": False,
    },
    "required_incomplete_items":
        required_incomplete,
    "files": artifact_rows,
}
write_json(
    ARTIFACT_MANIFEST_PATH,
    artifact_manifest,
)

checksum_files = [
    path
    for path in PACKAGE_ROOT.rglob("*")
    if path.is_file()
    and path != CHECKSUM_PATH
]
checksum_lines = [
    (
        f"{sha256_file(path)}  "
        f"{path.relative_to(PACKAGE_ROOT).as_posix()}"
    )
    for path in sorted(checksum_files)
]
CHECKSUM_PATH.write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8",
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as archive:
    for path in sorted(
        PACKAGE_ROOT.rglob("*")
    ):
        if not path.is_file():
            continue
        archive.write(
            path,
            arcname=(
                "NeuroFHIR_QC_AMIA_2026/"
                + path.relative_to(
                    PACKAGE_ROOT
                ).as_posix()
            ),
        )

if not ZIP_PATH.exists() or ZIP_PATH.stat().st_size == 0:
    raise AssertionError("Competition ZIP archive was not created.")

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise AssertionError(
            f"ZIP integrity failure: {bad_member}"
        )
    zip_member_count = len(archive.namelist())

print("=" * 112)
print("✅ Artifact manifest generated")
print(f"✅ SHA-256 inventory entries: {len(checksum_lines)}")
print(f"✅ ZIP members: {zip_member_count}")
print(f"✅ ZIP archive: {ZIP_PATH}")
print(
    f"📦 ZIP size: "
    f"{ZIP_PATH.stat().st_size / 1024 / 1024:.2f} MB"
)
print("=" * 112)

✅ Artifact manifest generated
✅ SHA-256 inventory entries: 61
✅ ZIP members: 62
✅ ZIP archive: /content/drive/MyDrive/neurofhir-qc/submission/NeuroFHIR_QC_AMIA_2026_Submission_Package.zip
📦 ZIP size: 0.48 MB


In [16]:
# Cell 10 — Final audit, manifest update, and release status

required_package_files = [
    FORM_DRAFT_JSON,
    FORM_DRAFT_MD,
    CHARACTER_AUDIT_CSV,
    RESOURCE_LIST_PATH,
    APP_CAPABILITY_PATH,
    PROVENANCE_CHECKLIST_CSV,
    MEASURED_RESULTS_CSV,
    MEASURED_RESULTS_MD,
    DEMO_SCRIPT_PATH,
    RUN_OF_SHOW_PATH,
    VIDEO_STORYBOARD_PATH,
    SLIDE_OUTLINE_PATH,
    SCREENSHOT_LIST_PATH,
    PORTAL_CLAIMS_PATH,
    CATEGORY_DECISION_PATH,
    STUDENT_ATTESTATION_TEMPLATE_PATH,
    NONSTUDENT_EVIDENCE_TEMPLATE_PATH,
    FINAL_CHECKLIST_PATH,
    PACKAGE_README_PATH,
    HTML_SUMMARY_PATH,
    ARTIFACT_MANIFEST_PATH,
    CHECKSUM_PATH,
    ZIP_PATH,
]

missing_or_empty = [
    str(path)
    for path in required_package_files
    if not path.exists() or path.stat().st_size == 0
]
if missing_or_empty:
    raise AssertionError(
        "Notebook 10 package artifacts are missing or empty:\n"
        + "\n".join(f" - {path}" for path in missing_or_empty)
    )

final_gate = {
    "notebook_09_completed":
        nb09_audit.get("status") == "completed",
    "submission_form_generated":
        FORM_DRAFT_MD.exists(),
    "all_character_limits_passed":
        not failed_character_fields,
    "capability_statement_generated":
        APP_CAPABILITY_PATH.exists(),
    "measured_results_generated":
        MEASURED_RESULTS_CSV.exists(),
    "provenance_completeness_100_percent":
        provenance_completeness_rate == 1.0,
    "demo_script_generated":
        DEMO_SCRIPT_PATH.exists(),
    "evidence_archive_generated":
        ZIP_PATH.exists(),
    "unsupported_claims_blocked": True,
    "category_eligibility_not_assumed": True,
}
failed_gate_items = [
    key
    for key, passed in final_gate.items()
    if not passed
]
if failed_gate_items:
    raise AssertionError(
        "Notebook 10 technical packaging gate failed: "
        + ", ".join(failed_gate_items)
    )

notebook_saved_in_drive = (
    NOTEBOOK_SAVE_PATH.exists()
    and NOTEBOOK_SAVE_PATH.stat().st_size > 0
)

release_status = (
    "portal-ready-draft"
    if portal_ready
    else "artifact-packaging-complete-not-portal-ready"
)

final_audit = {
    "project_name": "NeuroFHIR-QC",
    "project_version": project_config.get(
        "version",
        "0.1.0",
    ),
    "notebook_number": "10",
    "notebook_filename": NOTEBOOK_FILENAME,
    "status": "completed",
    "release_status": release_status,
    "audited_utc": utc_now(),
    "notebook_saved_in_drive": notebook_saved_in_drive,
    "official_submission": {
        "deadline": OFFICIAL_DEADLINE,
        "rules_url": OFFICIAL_RULES_URL,
        "presentation_date":
            OFFICIAL_PRESENTATION_DATE,
        "presentation_window":
            OFFICIAL_PRESENTATION_WINDOW,
        "presentation_length":
            OFFICIAL_PRESENTATION_LENGTH,
    },
    "technical_metrics": {
        "mean_whole_tumor_dice": mean_dice,
        "mean_absolute_volume_error_ml":
            mean_abs_volume_error_ml,
        "standard_perturbation_run_count":
            standard_perturbation_runs,
        "severe_challenge_run_count":
            challenge_runs,
        "longitudinal_scenario_alignment_rate":
            alignment_rate,
        "fhir_validation_pass_rate":
            validation_pass_count / validation_targets,
        "fhir_transaction_success_rate":
            transaction_success_count / transaction_count,
        "fhir_transaction_entry_success_rate":
            successful_entry_count / submitted_entry_count,
        "critical_field_preservation_rate":
            critical_field_preservation_rate,
        "review_transition_event_count":
            review_transition_event_count,
        "provenance_completeness_rate":
            provenance_completeness_rate,
    },
    "submission_readiness": {
        "selected_category": SUBMISSION_CATEGORY,
        "category_valid": category_is_valid,
        "category_eligibility_supported":
            category_eligibility_satisfied,
        "technical_package_complete":
            technical_package_complete,
        "administrative_status":
            administrative_status,
        "portal_ready": portal_ready,
        "required_incomplete_items":
            required_incomplete,
    },
    "scope": {
        "submission_form_draft_generated": True,
        "fhir_artifacts_packaged": True,
        "evaluation_evidence_packaged": True,
        "presentation_script_generated": True,
        "video_storyboard_generated": True,
        "slide_outline_generated": True,
        "screenshot_list_generated": True,
        "zip_archive_generated": True,
        "app_ui_deployed": bool(APP_OR_DEMO_URL),
        "competition_screenshots_captured": False,
        "human_usability_study_performed": bool(
            usability.get(
                "human_usability_study_performed",
                False,
            )
        ),
    },
    "safety": {
        "public_deidentified_imaging_only": True,
        "synthetic_fhir_only": True,
        "real_patient_data_packaged": False,
        "credentials_packaged": False,
        "clinical_validation_claimed": False,
        "clinical_deployment_claimed": False,
        "real_users_claimed": False,
        "patient_impact_claimed": False,
        "fabricated_usability_claims_blocked": True,
        "category_eligibility_fabrication_blocked": True,
    },
    "final_gate": final_gate,
    "output_paths": {
        "package_root":
            PACKAGE_ROOT.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        "zip_archive":
            ZIP_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        "submission_form":
            FORM_DRAFT_MD.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        "capability_statement":
            APP_CAPABILITY_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        "measured_results":
            MEASURED_RESULTS_CSV.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        "presentation_script":
            DEMO_SCRIPT_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        "final_checklist":
            FINAL_CHECKLIST_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
    },
    "next_step": (
        "Resolve category eligibility, deploy or record the final MVP, "
        "capture screenshots, add branding, verify administrative fields, "
        "and submit through the AMIA portal before the deadline."
    ),
}
write_json(AUDIT_JSON_PATH, final_audit)

AUDIT_MD_PATH.write_text(
    textwrap.dedent(
        f"""
        # Notebook 10 — Competition Artifacts

        **Status:** completed
        **Release status:** {release_status}
        **Audited:** {final_audit['audited_utc']}
        **Portal ready:** {portal_ready}

        ## Generated

        - official submission-form draft within character limits;
        - app CapabilityStatement and FHIR resource list;
        - measured-results and provenance-completeness tables;
        - eight-minute presentation script;
        - live-demo run of show;
        - video storyboard;
        - slide outline;
        - screenshot list;
        - category eligibility records;
        - verified claims and limitations;
        - evidence manifest and SHA-256 inventory;
        - competition ZIP archive.

        ## Required unresolved items

        {chr(10).join(f'- {row["item"]}: {row["next_action"]}' for row in required_incomplete) if required_incomplete else '- None'}

        ## Release archive

        `{ZIP_PATH.relative_to(PROJECT_ROOT).as_posix()}`

        ## Final rule

        Technical packaging is complete. Portal readiness remains false
        until every required administrative and eligibility item is
        truthfully resolved.
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

def notebook_entries(manifest: Any) -> list[dict[str, Any]]:
    if isinstance(manifest, list):
        return manifest
    if isinstance(manifest, dict):
        for key in ("notebooks", "entries", "workflow"):
            value = manifest.get(key)
            if isinstance(value, list):
                return value
        manifest["notebooks"] = []
        return manifest["notebooks"]
    raise ValueError(
        "Unrecognized notebook_manifest.json structure."
    )

entries = notebook_entries(notebook_manifest)
nb10_entry = next(
    (
        row
        for row in entries
        if str(
            row.get("notebook_number")
            or row.get("number")
            or ""
        ).zfill(2) == "10"
    ),
    None,
)
if nb10_entry is None:
    nb10_entry = {
        "notebook_number": "10",
        "filename": NOTEBOOK_FILENAME,
        "title": "Competition Artifacts",
    }
    entries.append(nb10_entry)

nb10_entry.update(
    {
        "status": "completed",
        "completed_utc":
            final_audit["audited_utc"],
        "release_status": release_status,
        "portal_ready": portal_ready,
        "selected_category":
            SUBMISSION_CATEGORY,
        "category_eligibility_supported":
            category_eligibility_satisfied,
        "package_zip":
            ZIP_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        "audit_path":
            AUDIT_JSON_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
    }
)
write_json(
    NOTEBOOK_MANIFEST_PATH,
    notebook_manifest,
)

print("=" * 112)
print("✅ Notebook 10 Competition Artifacts completed")
print(f"✅ Release status: {release_status}")
print("✅ Submission form and character audit completed")
print("✅ FHIR CapabilityStatement and evidence package completed")
print("✅ Eight-minute script, video storyboard, and slide outline completed")
print("✅ ZIP archive integrity passed")
print(f"📦 ZIP: {ZIP_PATH}")
print(f"📋 Portal ready: {portal_ready}")
if required_incomplete:
    print("⚠️ Final portal submission is blocked by:")
    for row in required_incomplete:
        print(f" - {row['item']}: {row['next_action']}")
else:
    print("✅ Required portal checklist is complete")
print(f"✅ Audit JSON: {AUDIT_JSON_PATH}")
print("📓 Manifest status: completed")
if not notebook_saved_in_drive:
    print(
        "⚠️ Save the executed notebook to "
        f"{NOTEBOOK_SAVE_PATH} and commit it to GitHub."
    )
print("➡️ Next: final UI/video, eligibility confirmation, branding, and portal submission")
print("=" * 112)

✅ Notebook 10 Competition Artifacts completed
✅ Release status: artifact-packaging-complete-not-portal-ready
✅ Submission form and character audit completed
✅ FHIR CapabilityStatement and evidence package completed
✅ Eight-minute script, video storyboard, and slide outline completed
✅ ZIP archive integrity passed
📦 ZIP: /content/drive/MyDrive/neurofhir-qc/submission/NeuroFHIR_QC_AMIA_2026_Submission_Package.zip
📋 Portal ready: False
⚠️ Final portal submission is blocked by:
 - Working app URL or reviewer-accessible video: Deploy the MVP UI or record/upload the demo
 - Submission category selected: Resolve Academic, Industry, or Student
 - Category eligibility evidence: Obtain advisor attestation for Student or truthful real-world-use evidence for non-student
 - Current affiliation verified: Enter current affiliation exactly
 - Conception date verified: Enter truthful conception month/date
 - Logo: Create and save logo image
 - Headshot: Add approved professional headshot
 - Promotional